# Model assignment and human review

Run this notebook from the repository root or its Notebooks directory. The six module cells regenerate the implementation in this checkout. The default run checks the code and prints the experimental design. Set the preflight or launch flag to run the solver.

In [ ]:
from pathlib import Path
import subprocess
import sys
from IPython.core.magic import register_cell_magic

ROOT = Path.cwd().resolve()
if ROOT.name == "Notebooks":
    ROOT = ROOT.parent
if not (ROOT / "Notebooks/formulation_runs.ipynb").is_file():
    raise RuntimeError("Open the notebook in its project checkout.")

PREFLIGHT = False
LAUNCH = False

@register_cell_magic
def writecode(relative_path, source):
    (ROOT / relative_path).write_text(source, encoding="utf-8")

In [ ]:
%%writecode Implementation/kp_analysis.py
"""Analysis of the campaign's saved results: tables, paired tests, the
expectations fixed before the run, the scaling fit, and figures.

    python kp_analysis.py [--runs ../runs] [--tables ../tables] [--figures ../figures] [--records ../Records]

Reads only runs/<block>/rep_<r>/<cell>.json with status ok; safe to run
while the campaign runs. Writes tables/<block>_summary.csv,
tables/paired.csv, tables/expectations.csv, tables/x7_scaling.csv,
Records/Expectations.md and figures/*.pdf.
"""
from __future__ import annotations

import argparse
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import optimize, stats

HERE = Path(__file__).resolve().parent
ROOT = HERE.parent
METRICS = ["p_systemic", "exp_loss", "cvar", "risk_obj", "wrong_per_run", "p_clean", "eff_models",
           "same_model_arc_share", "review_share_last_stage", "n_reviews", "n_used", "n_used_handoffs",
           "in_sample_risk_obj"]
SOLVER = ["solver_gap", "solver_time", "solver_status", "solver_time_to_target", "solver_bound", "solver_obj",
          "solver_time_limit"]


def load(runs: Path) -> tuple[pd.DataFrame, list]:
    """One row per (block, rep, cell, plan); and the list of job records."""
    rows, recs = [], []
    for f in sorted(runs.glob("X*/rep_*/*.json")):
        if f.name.endswith(".progress.json"):
            continue
        try:
            r = json.loads(f.read_text())
        except (OSError, ValueError):
            continue
        recs.append(r)
        if r.get("status") != "ok":
            continue
        res = r.get("result", {})
        for plan, d in res.get("plans", {}).items():
            row = dict(block=r["block"], rep=r["rep"], cell=r["cell"], plan=plan, set=r.get("set"))
            for k in ("rho", "tau", "q_diff", "copula", "dbar", "tie", "ell_mode", "kind", "L", "W", "S", "method"):
                if k in r:
                    row[k] = r[k]
            row["no_solution"] = bool(d.get("no_solution", False))
            for k in METRICS + SOLVER:
                if k in d:
                    row[k] = d[k]
            # A level first reached at the final bound update gets no callback time: a solve that
            # ended optimal (status 2) at gap g had reached every level at or above g by its end.
            done = d.get("solver_status") == 2 and isinstance(d.get("solver_gap"), (int, float))
            t_end = (d.get("solver_time") or 0.0) - (d.get("solver_build_time") or 0.0)
            if done and "solver_time" in d and row.get("solver_time_to_target") is None:
                row["solver_time_to_target"] = t_end
            ttg = d.get("solver_time_to_gap") or {}
            for g, t in ttg.items():
                if t is None and done and d["solver_gap"] <= float(g) + 1e-12:
                    t = t_end
                row[f"t_gap_{g}"] = t
            rows.append(row)
    return pd.DataFrame(rows), recs


def summary(df: pd.DataFrame) -> pd.DataFrame:
    keep = [m for m in METRICS + ["solver_gap", "solver_time"] if m in df]
    g = df[~df["no_solution"]].groupby(["block", "cell", "plan"])
    out = g[keep].mean()
    out["n"] = g.size()
    if "solver_gap" in df:
        out["share_gap_le_5pct"] = g["solver_gap"].apply(lambda s: float(np.mean(s.dropna() <= 0.05)) if s.notna().any() else np.nan)
    return out.reset_index()


def holm(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    run = 0.0
    m = len(p)
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (m - rank) * p[i]))
        adj[i] = run
    return adj


def paired(df: pd.DataFrame, block: str, base: str, other: str, metric: str = "p_systemic") -> pd.DataFrame:
    """Per cell: mean of (other - base) over replications, t interval, p value."""
    d = df[(df.block == block) & df.plan.isin([base, other]) & ~df.no_solution]
    piv = d.pivot_table(index=["cell", "rep"], columns="plan", values=metric)
    if base not in piv or other not in piv:
        return pd.DataFrame()
    diff = (piv[other] - piv[base]).dropna()
    rows = []
    for cell, s in diff.groupby(level=0):
        n = len(s)
        mean = float(s.mean())
        sd = float(s.std(ddof=1)) if n > 1 else float("nan")
        half = float(stats.t.ppf(0.975, n - 1) * sd / math.sqrt(n)) if n > 1 else float("nan")
        p = float(stats.ttest_1samp(s, 0.0).pvalue) if n > 1 and sd > 0 else float("nan")
        rows.append(dict(block=block, cell=cell, base=base, other=other, metric=metric, n=n, mean_diff=mean,
                         ci_low=mean - half, ci_high=mean + half, half_width=half, p=p))
    return pd.DataFrame(rows)


def tost(s: pd.Series, margin: float) -> float:
    """p value of two one-sided tests of |mean| < margin (equivalence)."""
    s = s.dropna()
    n = len(s)
    if n < 2 or s.std(ddof=1) == 0:
        return float("nan")
    se = s.std(ddof=1) / math.sqrt(n)
    p_low = 1 - stats.t.cdf((s.mean() + margin) / se, n - 1)
    p_high = stats.t.cdf((s.mean() - margin) / se, n - 1)
    return float(max(p_low, p_high))


def tobit_loglinear(y: np.ndarray, X: np.ndarray, cens: np.ndarray):
    """Right-censored normal regression of y on X (y censored at its value
    where cens is True). Returns coefficients, standard errors and sigma."""
    def nll(theta):
        b, ls = theta[:-1], theta[-1]
        sg = math.exp(ls)
        mu = X @ b
        z = (y - mu) / sg
        ll = np.where(cens, stats.norm.logsf(z), stats.norm.logpdf(z) - ls)
        return -float(ll.sum())
    b0 = np.linalg.lstsq(X, y, rcond=None)[0]
    th0 = np.append(b0, math.log(max(1e-3, float(np.std(y - X @ b0)))))
    res = optimize.minimize(nll, th0, method="BFGS")
    cov = res.hess_inv if hasattr(res, "hess_inv") else np.full((len(th0), len(th0)), np.nan)
    se = np.sqrt(np.clip(np.diag(cov), 0, None))
    return res.x[:-1], se[:-1], math.exp(res.x[-1]), bool(res.success)


def x7_scaling(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    d = df[(df.block == "X7") & df.plan.isin(["EF", "BD"])].copy()
    if d.empty:
        return d, {}
    n = d["L"] * d["W"]
    T = d["solver_time_to_target"]
    limit = d.get("solver_time_limit", pd.Series(np.nan, index=d.index)).fillna(d["solver_time"])
    cens = T.isna()
    d["n"] = n
    d["log_T"] = np.log(np.where(cens, limit.clip(lower=1.0), T.clip(lower=0.1)))
    d["censored"] = cens
    fits = {}
    for meth, g in d.groupby("plan"):
        if len(g) < 4 or g["n"].nunique() < 2:
            continue
        X = np.column_stack([np.ones(len(g)), g["n"].to_numpy(float), np.log(g["S"].to_numpy(float))])
        b, se, sg, ok = tobit_loglinear(g["log_T"].to_numpy(float), X, g["censored"].to_numpy(bool))
        fits[meth] = dict(b0=b[0], b1=b[1], b2=b[2], se_b1=se[1], se_b2=se[2], sigma=sg, converged=ok,
                          n_obs=len(g), censored_share=float(g["censored"].mean()))
    return d, fits


def expectations(df: pd.DataFrame, x7fits: dict) -> pd.DataFrame:
    """The deck's expectations X1 to X7 with the rule each is judged by."""
    rows = []

    def add(block, text, rule, value, passed):
        # passed: True, False, None (the data cannot judge the rule) or "tie" (no change to judge)
        outcome = ("not enough data" if passed is None else "no change" if passed == "tie"
                   else "pass" if passed else "fail")
        rows.append(dict(block=block, expectation=text, rule=rule, value=value, outcome=outcome))

    def mean(block, plan, metric, **cond):
        if metric not in df.columns or any(k not in df.columns for k in cond):
            return float("nan")                              # a block with no records yet
        d = df[(df.block == block) & (df.plan == plan) & ~df.no_solution]
        for k, v in cond.items():
            d = d[d[k] == v]
        return float(d[metric].mean()) if len(d) else float("nan")

    pr = paired(df, "X1", "ROUTE", "RA")
    base = pr[pr.cell == "rho0.3_tau0.15"] if len(pr) else pr
    if len(base):
        hw = 100 * float(base.half_width.iloc[0])
        add("X1", "Base-cell 95% half-width of RA minus ROUTE systemic failure falls to about 0.67 points",
            "half-width at most 1.0 point", f"{hw:.2f} points (n={int(base.n.iloc[0])})",
            None if math.isnan(hw) else hw <= 1.0)
    else:
        add("X1", "Base-cell half-width", "half-width at most 1.0 point", "", None)
    x1 = df[(df.block == "X1") & df.plan.isin(["RA", "IND"]) & ~df.no_solution]
    if len(x1):
        piv = x1.pivot_table(index=["cell", "rep"], columns="plan", values="p_systemic")
        if "IND" in piv:
            ps = [(cell, tost((s["IND"] - s["RA"]).dropna(), 0.01)) for cell, s in piv.groupby(level=0)]
            ps = [(c, p) for c, p in ps if not math.isnan(p)]
            ok = all(p < 0.05 for _, p in ps) if ps else None
            add("X1", "Planning under independent errors is equivalent to planning under correlation (H2)",
                "TOST p < 0.05 at a 1-point margin in every cell with rho > 0",
                "; ".join(f"{c}: p={p:.3f}" for c, p in ps), ok)
    for cop in ("gaussian", "t"):
        lo = mean("X2", "RA-PAIR", "eff_models", copula=cop, q_diff=min(df.get("q_diff", pd.Series([0.45])).dropna(), default=0.45))
        hi = mean("X2", "RA-PAIR", "eff_models", copula=cop, q_diff=max(df.get("q_diff", pd.Series([0.95])).dropna(), default=0.95))
        add("X2", f"RA-PAIR mixes models less when different models pass more errors ({cop})",
            "mean effective models at the highest q_diff below that at the lowest", f"{hi:.2f} vs {lo:.2f}",
            None if math.isnan(lo) or math.isnan(hi) else hi < lo)
    adv = {}
    for cop in ("gaussian", "t"):
        adv[cop] = mean("X2", "ROUTE", "p_systemic", copula=cop) - mean("X2", "RA-PAIR", "p_systemic", copula=cop)
    add("X2", "RA-PAIR's advantage over ROUTE grows under the t copula",
        "mean ROUTE minus RA-PAIR systemic failure larger under t", f"t {adv['t']:.4f} vs gaussian {adv['gaussian']:.4f}",
        None if any(math.isnan(v) for v in adv.values()) else adv["t"] > adv["gaussian"])
    e_lo = mean("X3", "RA-PAIR", "eff_models", dbar=0.6)
    e_hi = mean("X3", "RA-PAIR", "eff_models", dbar=1.0)
    add("X3", "RA-PAIR uses more models as review success falls", "mean effective models at dbar 0.6 above dbar 1.0",
        f"{e_lo:.2f} vs {e_hi:.2f}", None if math.isnan(e_lo) or math.isnan(e_hi) else e_lo > e_hi)
    r_del = mean("X4", "RA-PAIR", "review_share_last_stage", ell_mode="deliverables")
    r_all = mean("X4", "RA-PAIR", "review_share_last_stage", ell_mode="all")
    add("X4", "A larger share of reviews sits in the last stage when only deliverables count",
        "mean last-stage review share higher under deliverable loss", f"{r_del:.3f} vs {r_all:.3f}",
        None if math.isnan(r_del) or math.isnan(r_all) else r_del > r_all)
    caps = sorted({p for p in df[df.block == "X5"].plan.unique() if p.startswith("RA-CAP")},
                  key=lambda p: float(p.replace("RA-CAP", "")))
    if caps:
        w = [mean("X5", p, "wrong_per_run") for p in caps]
        add("X5", "Expected wrong outputs rise as the cap tightens", "mean wrong outputs per run decrease in beta",
            "; ".join(f"{p}: {v:.3f}" for p, v in zip(caps, w)),
            None if any(math.isnan(v) for v in w) else "tie" if max(w) - min(w) < 1e-12
            else all(w[i] >= w[i + 1] - 1e-12 for i in range(len(w) - 1)))
    for kind in ("tree", "random"):
        h0 = mean("X6", "RA-PAIR", "n_used_handoffs", kind=kind, rho=0.0)
        h6 = mean("X6", "RA-PAIR", "n_used_handoffs", kind=kind, rho=0.6)
        add("X6", f"Chosen decompositions have fewer handoffs as rho rises ({kind})",
            "mean used handoffs at rho 0.6 below rho 0", f"{h6:.2f} vs {h0:.2f}",
            None if math.isnan(h0) or math.isnan(h6) else h6 < h0)
    for meth, f in x7fits.items():
        lo = f["b1"] - 1.96 * f["se_b1"]
        add("X7", f"Time to a 1% gap grows exponentially in tasks ({meth})",
            "censored-regression slope on tasks with 95% interval above 0",
            f"b1={f['b1']:.4f} (se {f['se_b1']:.4f}), censored {f['censored_share']:.0%}, "
            f"{'converged' if f['converged'] else 'not converged'}",
            # a fit that did not converge, or rests on almost only censored times, cannot judge the slope
            None if math.isnan(lo) or not f["converged"] or f["censored_share"] > 0.9 else lo > 0)
    return pd.DataFrame(rows)


def figures(df: pd.DataFrame, x7: pd.DataFrame, out: Path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    out.mkdir(parents=True, exist_ok=True)
    d = df[(df.block == "X1") & ~df.no_solution]
    if len(d):
        cells = sorted(d.cell.unique())
        plans = [p for p in ["ROUTE", "EV", "IND", "RN", "RA"] if p in d.plan.unique()]
        fig, ax = plt.subplots(figsize=(10, 3.6))
        w = 0.8 / max(1, len(plans))
        for k, p in enumerate(plans):
            m = d[d.plan == p].groupby("cell")["p_systemic"].mean().reindex(cells)
            ax.bar(np.arange(len(cells)) + k * w, 100 * m.values, w, label=p)
        ax.set_xticks(np.arange(len(cells)) + 0.4 - w / 2)
        ax.set_xticklabels([c.replace("_", "\n") for c in cells], fontsize=8)
        ax.set_ylabel("systemic failure (% of runs)")
        ax.legend(ncol=len(plans), fontsize=8, frameon=False)
        fig.tight_layout()
        fig.savefig(out / "x1_systemic.pdf")
        plt.close(fig)
    if len(x7):
        fig, ax = plt.subplots(figsize=(6, 3.6))
        for (meth, S), g in x7.groupby(["plan", "S"]):
            m = g.groupby("n")["log_T"].mean()
            ax.plot(m.index, np.exp(m.values), marker="o", label=f"{meth}, {S} scenarios")
        ax.set_yscale("log")
        ax.set_xlabel("tasks")
        ax.set_ylabel("seconds to a 1% gap (limit if censored)")
        ax.legend(fontsize=7, frameon=False)
        fig.tight_layout()
        fig.savefig(out / "x7_scaling.pdf")
        plt.close(fig)


def write_all(runs: Path, tables: Path, figs: Path | None, records: Path | None) -> dict:
    df, recs = load(runs)
    tables.mkdir(parents=True, exist_ok=True)
    status = pd.Series([r.get("status") for r in recs]).value_counts().to_dict()
    if df.empty:
        return dict(rows=0, status=status)
    summ = summary(df)
    for b, g in summ.groupby("block"):
        g.to_csv(tables / f"{b}_summary.csv", index=False)
    comps = [("X1", "ROUTE", "RA"), ("X1", "EV", "RA"), ("X1", "IND", "RA"), ("X1", "RN", "RA"),
             ("X2", "ROUTE", "RA-PAIR"), ("X2", "RA", "RA-PAIR"), ("X3", "ROUTE", "RA-PAIR"), ("X3", "RA", "RA-PAIR"),
             ("X4", "ROUTE", "RA-PAIR"), ("X4", "RA", "RA-PAIR"), ("X6", "ROUTE", "RA-PAIR")]
    pr = pd.concat([paired(df, *c) for c in comps], ignore_index=True) if comps else pd.DataFrame()
    if len(pr):
        pr["p_holm"] = np.nan
        for b, idx in pr.groupby("block").groups.items():
            ok = pr.loc[idx, "p"].notna()
            pr.loc[idx[ok.values], "p_holm"] = holm(pr.loc[idx[ok.values], "p"].to_numpy())
        pr.to_csv(tables / "paired.csv", index=False)
    x7, fits = x7_scaling(df)
    if len(x7):
        x7.to_csv(tables / "x7_scaling.csv", index=False)
    exp = expectations(df, fits)
    exp.to_csv(tables / "expectations.csv", index=False)
    if records is not None:
        records.mkdir(parents=True, exist_ok=True)
        lines = ["# Expectations fixed before the run, and their outcomes", "",
                 f"Computed by `Implementation/kp_analysis.py` from {len(recs)} job records ({status}).", "",
                 "| Block | Expectation | Rule | Value | Outcome |", "| --- | --- | --- | --- | --- |"]
        for _, r in exp.iterrows():
            lines.append(f"| {r.block} | {r.expectation} | {r.rule} | {r.value} | {r.outcome} |")
        (records / "Expectations.md").write_text("\n".join(lines) + "\n")
    if figs is not None:
        figures(df, x7, figs)
    return dict(rows=len(df), status=status, expectations=len(exp), x7_fits=fits)


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--runs", default=str(ROOT / "runs"))
    ap.add_argument("--tables", default=str(ROOT / "tables"))
    ap.add_argument("--figures", default=str(ROOT / "figures"))
    ap.add_argument("--records", default=str(ROOT / "Records"))
    ap.add_argument("--no-figures", action="store_true")
    a = ap.parse_args(argv)
    out = write_all(Path(a.runs), Path(a.tables), None if a.no_figures else Path(a.figures), Path(a.records))
    print(json.dumps(out, default=str))


if __name__ == "__main__":
    main()


In [ ]:
%%writecode Implementation/kp_blocks.py
"""Blocks X1 to X7 of the formulation deck's Mac Studio protocol: the job
list, and one job's plans, warm starts and scores.

A job is one cell of one replication; a set is all jobs of one block and
replication, the unit the monitor commits. Seeds follow the manuscript:
layered workflow seed = replication, training sample 1000 + replication,
test sample 900000 + replication, so cells of one replication share the
workflow and common random numbers, and X1 replications 1 to 8 rebuild the
manuscript's E1 workflows.
"""
from __future__ import annotations

import copy
import math
import time

import numpy as np

import kp_cuts as kc
import kp_model as km

DEFAULTS = dict(
    lam=0.5, alpha=0.9, gamma=0.3, q_same=0.9, q_diff=0.5, fan=2.0, budget_share=0.6,
    tau=0.15, rho=0.3, L=8, W=5, p_scale=1.0, e_sys_share=0.25,
    train_S=200, test_S=50000, ev_time_limit=60.0, ls_seconds=30.0,
    # Stopping: a solve ends at a proven gap of mip_gap (5 percent; 1 percent in
    # the scaling study X7) or at time_limit, whichever comes first. time_limit
    # None lets kp_campaign compute it from the campaign budget.
    mip_gap=0.05, x7_mip_gap=0.01, time_limit=None, time_limit_max=1800.0, time_limit_min=120.0,
    # Budget: the whole campaign ends within campaign_hours of its first launch,
    # with reserve_hours kept for the last jobs and the final records.
    campaign_hours=48.0, reserve_hours=2.0, efficiency=0.85, job_overhead=40.0,
    threads=2, soft_mem_gb=None, nodefile_start=None, mip_focus=None,
    reps=dict(X1=41, X2=10, X3=10, X4=10, X5=10, X6=10, X7=3),
    blocks=["X1", "X2", "X3", "X4", "X5", "X6", "X7"],
    x2_q_same=0.70, x2_q_diff=[0.45, 0.70, 0.95], x2_copulas=["gaussian", "t"], t_nu=4.0,
    x3_dbar=[0.6, 0.8, 1.0], x3_tie=0.5,
    x5_beta=[0.02, 0.01, 0.005],
    x6_kinds=["tree", "random"], x6_rho=[0.0, 0.3, 0.6],
    x7_sizes=[[8, 5], [10, 6], [10, 8], [12, 10]], x7_S=[200, 400, 800, 1600, 3200],
    x7_methods=["EF", "BD"], x7_pi_S=[200, 400, 800], x7_pi_size=[8, 5], x7_chain=False,
)


def config(overrides: dict | None = None) -> dict:
    c = copy.deepcopy(DEFAULTS)
    for k, v in (overrides or {}).items():
        if k == "reps" and isinstance(v, dict):
            c["reps"].update(v)
        else:
            c[k] = v
    if c["time_limit"] is None:
        c["time_limit"] = c["time_limit_max"]
    return c


def preflight_config() -> dict:
    """Tiny settings that run every block end to end in minutes."""
    return config(dict(L=4, W=3, p_scale=3.0, train_S=40, test_S=2000, time_limit=20.0, ev_time_limit=10.0,
                       ls_seconds=2.0, reps={b: 1 for b in DEFAULTS["reps"]}, campaign_hours=1.0,
                       reserve_hours=0.05, time_limit_min=5.0,
                       x7_sizes=[[4, 3], [5, 3]], x7_S=[30, 60], x7_pi_S=[30], x7_pi_size=[4, 3]))


def job_seconds(job: dict, c: dict, time_limit: float | None = None) -> float:
    """Worst-case wall time of a job: every solve at its time limit."""
    T = c["time_limit"] if time_limit is None else time_limit
    ls = min(c["ls_seconds"], 0.1 * T)
    return c["job_overhead"] + c["ev_time_limit"] * job["n_ev"] + job["n_mip"] * (T + ls)


def _job(block, rep, cell, n_mip, n_ev=1, **kw):
    j = dict(block=block, rep=int(rep), cell=cell, n_mip=int(n_mip), n_ev=int(n_ev), **kw)
    j["id"] = f"{block}/rep_{rep:03d}/{cell}"
    j["set"] = f"{block}/rep_{rep:03d}"
    return j


def jobs_for(block: str, rep: int, c: dict) -> list:
    out = []
    if block == "X1":
        for rho in (0.0, 0.3, 0.6):
            for tau in (0.05, 0.15, 0.30):
                out.append(_job("X1", rep, f"rho{rho:.1f}_tau{tau:.2f}", 2 if rho == 0 else 3, rho=rho, tau=tau))
    elif block == "X2":
        for qd in c["x2_q_diff"]:
            for cop in c["x2_copulas"]:
                out.append(_job("X2", rep, f"qdiff{qd:.2f}_{cop}", 2, q_diff=qd, copula=cop))
    elif block == "X3":
        for d in c["x3_dbar"]:
            for tie in (["independent", "G0"] if d < 1 else ["independent"]):
                out.append(_job("X3", rep, f"dbar{d:.1f}_{tie}", 2, dbar=d, tie=tie))
    elif block == "X4":
        for mode in ("all", "deliverables"):
            out.append(_job("X4", rep, f"loss_{mode}", 1 if mode == "all" else 2, ell_mode=mode))
    elif block == "X5":
        out.append(_job("X5", rep, "caps", 1 + len(c["x5_beta"])))
    elif block == "X6":
        for kind in c["x6_kinds"]:
            for rho in c["x6_rho"]:
                out.append(_job("X6", rep, f"{kind}_rho{rho:.1f}", 1, kind=kind, rho=rho))
    elif block == "X7":
        for (L, W) in c["x7_sizes"]:
            for S in c["x7_S"]:
                for meth in c["x7_methods"]:
                    out.append(_job("X7", rep, f"n{L * W:03d}_S{S:04d}_{meth}", 1, n_ev=0, L=L, W=W, S=S, method=meth))
        L, W = c["x7_pi_size"]
        for S in c["x7_pi_S"]:
            out.append(_job("X7", rep, f"n{L * W:03d}_S{S:04d}_PI", 1, n_ev=0, L=L, W=W, S=S, method="PI"))
    else:
        raise ValueError(block)
    for j in out:
        j["wc"] = job_seconds(j, c)
    return out


def budget(c: dict, workers: int, blocks=None, hours: float | None = None) -> dict:
    """Time limit per solve that fits the campaign in its wall-clock budget.

    Capacity is workers x (hours - reserve) x efficiency job-seconds. Every
    solve gets the same limit T, from capacity = sum over jobs of
    (overhead + EV + n_mip (T + local search)). When T falls below
    time_limit_min, replications are cut, X1 first down to 8, then the other
    blocks down to 5 and X7 down to 2, until T reaches the minimum."""
    c = copy.deepcopy(c)
    hours = c["campaign_hours"] if hours is None else hours
    cap = max(0.0, (hours - c["reserve_hours"]) * 3600.0 * workers * c["efficiency"])
    floors = dict(X1=8, X7=2)
    cuts = []

    def solve_T(cc):
        q = build_queue(cc, blocks)
        fixed = sum(cc["job_overhead"] + cc["ev_time_limit"] * j["n_ev"] for j in q)
        mips = sum(j["n_mip"] for j in q)
        # T + min(ls, 0.1 T) per solve: solve for T with ls at its cap, then check
        T = (cap - fixed) / max(1, mips) - cc["ls_seconds"]
        if T < 10 * cc["ls_seconds"]:
            T = (cap - fixed) / max(1, mips) / 1.1
        return T, q

    T, q = solve_T(c)
    order = ["X1", "X2", "X3", "X4", "X5", "X6", "X7"]
    while T < c["time_limit_min"]:
        cand = [b for b in order if b in (blocks or c["blocks"]) and c["reps"][b] > floors.get(b, 5)]
        if not cand:
            break
        b = cand[0] if "X1" in cand and c["reps"]["X1"] > 8 else max(cand, key=lambda x: c["reps"][x])
        c["reps"][b] -= 1
        cuts.append(b)
        T, q = solve_T(c)
    T = float(min(c["time_limit_max"], max(c["time_limit_min"], T)))
    c["time_limit"] = T
    q = build_queue(c, blocks)
    worst = sum(j["wc"] for j in q)
    return dict(time_limit=T, reps=c["reps"], cuts=cuts, jobs=len(q), solves=sum(j["n_mip"] for j in q),
                worst_job_hours=worst / 3600.0, worst_wall_hours=worst / 3600.0 / workers / c["efficiency"],
                capacity_job_hours=cap / 3600.0, config=c)


def build_queue(c: dict, blocks=None) -> list:
    """Replication-major order: replication 1 of every block, then
    replication 2, so early sets cover every block."""
    blocks = blocks or c["blocks"]
    top = max(c["reps"][b] for b in blocks)
    q = []
    for rep in range(1, top + 1):
        for b in blocks:
            if rep <= c["reps"][b]:
                q += jobs_for(b, rep, c)
    return q


# ----------------------------------------------------------------- one job
def _q(c, same=None, diff=None):
    return km.q_matrix(c["q_same"] if same is None else same, c["q_diff"] if diff is None else diff)


def _opts(c, log_file=None):
    return dict(threads=c["threads"], mip_gap=c["mip_gap"], soft_mem_gb=c.get("soft_mem_gb"),
                nodefile_start=c.get("nodefile_start"), mip_focus=c.get("mip_focus"), log_file=log_file)


class Planner:
    """Plans of one job on one instance with a shared warm-start pool."""

    def __init__(self, inst, train, test, c, env, log_file=None, progress=None, job=None):
        import kp_solve as ks
        self.ks, self.inst, self.train, self.test, self.c, self.env = ks, inst, train, test, c, env
        self.opts = _opts(c, log_file)
        self.progress = progress
        job = job or {}
        self.deadline = job.get("deadline")           # epoch seconds the job must end by
        self.mips_left = max(1, int(job.get("n_mip", 1)))
        if job.get("block") == "X7":
            self.opts["mip_gap"] = c["x7_mip_gap"]
        self.pool = []                       # (name, a, y) candidates for warm starts
        self.plans = {}

    def score(self, name, a, y, solver=None):
        """Scores on the test sample and the job's training sample, both with
        the true distribution and loss weights."""
        d = km.metrics(self.inst, a, y, self.test, self.c["lam"], self.c["alpha"])
        d["in_sample_risk_obj"] = km.objective(self.inst, a, y, self.train, self.c["lam"], self.c["alpha"])
        d["a"], d["y"] = [int(v) for v in a], [int(v) for v in y]
        if solver:
            d.update({f"solver_{k}": v for k, v in solver.items() if k not in ("a", "y")})
        self.plans[name] = d
        self.pool.append((name, np.asarray(a, int), np.asarray(y, bool)))
        return d

    def best_start(self, sc, lam, cap_beta=None, ls=True, inst=None):
        """Warm start: the pool candidate with the best objective on the sample
        being solved, improved by local search; with a cap, a candidate that
        meets it on that sample is preferred."""
        inst = inst or self.inst
        alpha = self.c["alpha"]

        def val(a, y):
            e = km.propagate(inst, a, y, sc)
            v = km.objective(inst, a, y, sc, lam, alpha, e=e)
            if cap_beta is not None:
                v += 1e3 * max(0.0, float(np.mean(e.sum(1) >= inst.e_sys)) - cap_beta)
            return v

        same_use = [(n, a, y) for n, a, y in self.pool]
        name, a, y = min(same_use, key=lambda t: val(t[1], t[2]))
        info = dict(start_from=name, start_value=val(a, y))
        if ls and self.c["ls_seconds"] > 0:
            a, y, v, st = kc.local_search(inst, sc, a, y, lam, alpha, self.c["ls_seconds"], cap_beta=cap_beta)
            info.update(st)
        return (a, y), info

    def route(self):
        a, y = self.ks.solve_route(self.inst, self.env, threads=min(2, self.c["threads"]))
        return self.score("ROUTE", a, y)

    def ev(self, q):
        name, a0, y0 = self.pool[0]
        a, y, st = self.ks.solve_ev(self.inst, q, self.env, self.opts, self.c["ev_time_limit"], start=(a0, y0))
        return self.score("EV", a, y, st)

    def ef(self, name, sc, lam, cap_beta=None, per_scenario_reviews=False, method="EF", plan_inst=None):
        """Solve on sample sc (and plan_inst, when the plan assumes other loss
        weights), then score the plan under the true instance."""
        pin = plan_inst or self.inst
        T = float(self.c["time_limit"])
        if self.deadline:                            # share the job's remaining time
            T = min(T, (self.deadline - time.time() - 20.0) / self.mips_left)
        T = max(10.0, T)
        self.mips_left = max(1, self.mips_left - 1)
        ls_keep = self.c["ls_seconds"]
        self.c = dict(self.c, ls_seconds=min(ls_keep, 0.1 * T))
        start, info = self.best_start(sc, lam, cap_beta, inst=pin)
        self.c = dict(self.c, ls_seconds=ls_keep)
        T = T - info.get("ls_seconds", 0.0)
        if method == "BD":
            res = self.ks.solve_benders(pin, sc, lam, self.c["alpha"], self.env, self.opts, T, start=start,
                                        label=name, progress_path=self.progress)
        else:
            res = self.ks.solve_ef(pin, sc, lam, self.c["alpha"], self.env, self.opts, T, start=start,
                                   cap_beta=cap_beta, per_scenario_reviews=per_scenario_reviews, label=name,
                                   progress_path=self.progress)
        res.update(info)
        if "a" not in res:
            # the PI bound has no plan by design; otherwise no incumbent (infeasible cap, time or
            # memory limit). Either way keep the solver record.
            self.plans[name] = dict(no_solution=not per_scenario_reviews, bound_only=bool(per_scenario_reviews),
                                    **{f"solver_{k}": v for k, v in res.items()})
            return res
        res["solve_sample_obj"] = km.objective(pin, res["a"], res["y"], sc, lam, self.c["alpha"])
        return self.score(name, res["a"], res["y"], res)


def _layered(c, rep, **kw):
    return km.make_instance(c["L"], c["W"], c["fan"], c["budget_share"], kw.pop("tau", c["tau"]), seed=rep,
                            p_scale=c["p_scale"], e_sys_share=c["e_sys_share"], **kw)


def run_job(job: dict, c: dict, env, log_file=None, progress=None) -> dict:
    t0 = time.time()
    b, rep = job["block"], job["rep"]
    lam, gam = c["lam"], c["gamma"]
    tr_seed, te_seed = 1000 + rep, 900000 + rep
    S, St = c["train_S"], c["test_S"]
    out = dict(cell=job["cell"])
    if b == "X1":
        rho = job["rho"]
        inst = _layered(c, rep, tau=job["tau"])
        q = _q(c)
        train = km.sample_scenarios(inst, S, rho, gam, q, tr_seed)
        test = km.sample_scenarios(inst, St, rho, gam, q, te_seed)
        P = Planner(inst, train, test, c, env, log_file, progress, job)
        P.route()
        P.ev(q)
        P.ef("RN", train, 0.0)
        if rho > 0:
            ind = km.sample_scenarios(inst, S, 0.0, gam, q, tr_seed)
            P.ef("IND", ind, lam)
        P.ef("RA", train, lam)
    elif b == "X2":
        inst = _layered(c, rep)
        q_true = _q(c, c["x2_q_same"], job["q_diff"])
        cop = job["copula"]
        train = km.sample_scenarios(inst, S, c["rho"], gam, q_true, tr_seed, copula=cop, nu=c["t_nu"])
        test = km.sample_scenarios(inst, St, c["rho"], gam, q_true, te_seed, copula=cop, nu=c["t_nu"])
        assumed = km.sample_scenarios(inst, S, c["rho"], gam, _q(c), tr_seed)     # manuscript's assumption
        P = Planner(inst, train, test, c, env, log_file, progress, job)
        P.route()
        P.ev(_q(c))
        P.ef("RA", assumed, lam)
        P.ef("RA-PAIR", train, lam)
    elif b == "X3":
        inst = _layered(c, rep, dbar=job["dbar"])
        q = _q(c)
        tie = None if job["tie"] == "independent" else c["x3_tie"]
        train = km.sample_scenarios(inst, S, c["rho"], gam, q, tr_seed, review_tie=tie)
        test = km.sample_scenarios(inst, St, c["rho"], gam, q, te_seed, review_tie=tie)
        perfect = _layered(c, rep, dbar=1.0)
        assumed = km.sample_scenarios(perfect, S, c["rho"], gam, q, tr_seed)       # reviews assumed perfect
        P = Planner(inst, train, test, c, env, log_file, progress, job)
        P.route()
        P.ev(q)
        P.ef("RA", assumed, lam)
        P.ef("RA-PAIR", train, lam)
    elif b == "X4":
        inst = _layered(c, rep, ell_mode=job["ell_mode"])
        q = _q(c)
        train = km.sample_scenarios(inst, S, c["rho"], gam, q, tr_seed)
        test = km.sample_scenarios(inst, St, c["rho"], gam, q, te_seed)
        P = Planner(inst, train, test, c, env, log_file, progress, job)
        P.route()
        P.ev(q)
        if job["ell_mode"] == "deliverables":
            P.ef("RA", train, lam, plan_inst=_layered(c, rep))   # plans with every output weighted 1
        P.ef("RA-PAIR", train, lam)
    elif b == "X5":
        inst = _layered(c, rep)
        q = _q(c)
        train = km.sample_scenarios(inst, S, c["rho"], gam, q, tr_seed)
        test = km.sample_scenarios(inst, St, c["rho"], gam, q, te_seed)
        P = Planner(inst, train, test, c, env, log_file, progress, job)
        P.route()
        P.ev(q)
        P.ef("RA-PAIR", train, lam)
        for beta in sorted(c["x5_beta"], reverse=True):        # looser cap first: its plan starts the next
            P.ef(f"RA-CAP{beta:g}", train, lam, cap_beta=beta)
        out["betas"] = sorted(c["x5_beta"], reverse=True)
    elif b == "X6":
        inst = km.make_template(job["kind"], rep, c["budget_share"], c["tau"], c["p_scale"],
                                e_sys_share=c["e_sys_share"])
        q = _q(c)
        train = km.sample_scenarios(inst, S, job["rho"], gam, q, tr_seed)
        test = km.sample_scenarios(inst, St, job["rho"], gam, q, te_seed)
        P = Planner(inst, train, test, c, env, log_file, progress, job)
        P.route()
        P.ev(q)
        P.ef("RA-PAIR", train, lam)
        out["template"] = dict(n=inst.n, nA=inst.nA, and_groups=len(inst.and_groups),
                               or_groups=len(inst.or_groups), e_sys=inst.e_sys, B=inst.B, H=inst.H)
    elif b == "X7":
        inst = km.make_instance(job["L"], job["W"], c["fan"], c["budget_share"], c["tau"], seed=100 + rep,
                                p_scale=c["p_scale"], e_sys_share=c["e_sys_share"])
        q = _q(c)
        sc = km.sample_scenarios(inst, job["S"], c["rho"], gam, q, 2000 + rep)
        test = km.sample_scenarios(inst, min(St, 20000), c["rho"], gam, q, te_seed)
        P = Planner(inst, sc, test, c, env, log_file, progress, job)
        P.route()
        if job["method"] == "PI":
            res = P.ef("PI", sc, lam, per_scenario_reviews=True)
            out["pi"] = {k: v for k, v in res.items() if k not in ("a", "y", "trace")}
            out["pi"]["trace"] = res.get("trace", [])
        else:
            P.ef(job["method"], sc, lam, method=job["method"])
        out["censored"] = {k: (v.get("solver_time_to_target") is None) for k, v in P.plans.items() if k != "ROUTE"}
    else:
        raise ValueError(b)
    out.update(n=P.inst.n, nA=P.inst.nA, B=P.inst.B, H=P.inst.H, e_sys=P.inst.e_sys, plans=P.plans,
               wall=time.time() - t0)
    return out


def fake_run_job(job: dict, c: dict, env=None, log_file=None, progress=None) -> dict:
    """Stand-in for tests of the scheduler and monitors: no solver."""
    time.sleep(float(c.get("fake_seconds", 0.05)))
    if job.get("cell", "").endswith(c.get("fake_fail_suffix", "\0")):
        raise RuntimeError("fake failure")
    return dict(cell=job["cell"], plans=dict(ROUTE=dict(p_systemic=0.1), RA=dict(p_systemic=0.02)), wall=0.0)


In [ ]:
%%writecode Implementation/kp_campaign.py
"""Parallel campaign runner for the Mac Studio.

    python kp_campaign.py plan     [--workers J] [--threads T] [--blocks X1,X7]
    python kp_campaign.py launch   [--workers J] [--threads T] [--blocks ...] [--retry-failed] [--detach]
    python kp_campaign.py preflight                # every block at tiny size, into runs/_preflight
    python kp_campaign.py status
    python kp_campaign.py stop

J worker processes each hold one Gurobi environment (one license session for
the whole campaign) and run one job at a time with T solver threads. The
default is T = 2 and J = (physical cores - 2) // T, the rule of the BiObj
Mac Studio campaigns: 11 workers x 2 threads on its 24 cores, with two cores
left for the notebook, the monitors and macOS. A job is sent to an idle
worker only when the free memory covers the per-job allowance, which is also
the solver's soft memory limit.

The campaign ends within campaign_hours (48) of its first launch. At the
first launch the time limit per solve is computed so that every job fits in
that budget (kp_blocks.budget); a relaunch keeps the first launch's deadline
and plan. Each job receives a deadline and divides its remaining time among
its solves, and a job that can no longer finish before the campaign deadline
is recorded as skipped. A relaunch after the deadline closes the campaign:
every unfinished job is recorded as skipped and CAMPAIGN_DONE.json is
written, so it never starts a second budget without --new-campaign.
`launch --detach` starts the scheduler in its own session under caffeinate
and returns, so a relaunch from a monitor or an agent outlives its caller.
`stop` writes runs/STOPPED.json, which tells the monitor not to relaunch;
the next launch removes it.
Every finished job writes runs/<block>/rep_<r>/<cell>.json atomically, so a
stopped campaign resumes where it stopped: finished jobs are skipped, a job
whose worker died is retried once, and a failed job reruns only with
--retry-failed. runs/status.json is rewritten every 10 seconds for the
monitors; runs/run.lock holds the scheduler's process id while it runs.
"""
from __future__ import annotations

import argparse
import datetime as dt
import json
import multiprocessing as mp
import os
import platform
import queue as queue_mod
import shutil
import signal
import socket
import subprocess
import sys
import time
import traceback
from pathlib import Path

HERE = Path(__file__).resolve().parent
ROOT = HERE.parent
RUNS = ROOT / "runs"


def now() -> str:
    return dt.datetime.now().isoformat(timespec="seconds")


def write_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + f".tmp{os.getpid()}")
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
    os.replace(tmp, path)


def read_json(path: Path, default=None):
    try:
        with open(path) as f:
            return json.load(f)
    except (OSError, ValueError):
        return default


def pid_alive(pid) -> bool:
    try:
        os.kill(int(pid), 0)
        return True
    except (OSError, TypeError, ValueError):
        return False


def _sysctl(name):
    try:
        return subprocess.run(["sysctl", "-n", name], capture_output=True, text=True, timeout=5).stdout.strip()
    except Exception:
        return ""


def machine() -> dict:
    d = dict(host=socket.gethostname(), system=platform.system(), machine=platform.machine(),
             python=platform.python_version(), logical_cpus=os.cpu_count())
    if platform.system() == "Darwin":
        d["chip"] = _sysctl("machdep.cpu.brand_string")
        d["physical_cpus"] = int(_sysctl("hw.physicalcpu") or 0) or os.cpu_count()
        d["performance_cpus"] = int(_sysctl("hw.perflevel0.physicalcpu") or 0) or None
        d["efficiency_cpus"] = int(_sysctl("hw.perflevel1.physicalcpu") or 0) or None
        d["memory_gb"] = round(int(_sysctl("hw.memsize") or 0) / 2 ** 30, 1)
    else:
        d["physical_cpus"] = os.cpu_count()
        try:
            for ln in open("/proc/cpuinfo"):
                if ln.startswith("model name"):
                    d["chip"] = ln.split(":", 1)[1].strip()
                    break
            for ln in open("/proc/meminfo"):
                if ln.startswith("MemTotal"):
                    d["memory_gb"] = round(int(ln.split()[1]) / 2 ** 20, 1)
        except OSError:
            pass
    try:
        import gurobipy as gp
        d["gurobi"] = ".".join(str(v) for v in gp.gurobi.version())
    except Exception as ex:
        d["gurobi"] = f"unavailable: {ex!r}"
    try:
        import numpy, scipy
        d["numpy"], d["scipy"] = numpy.__version__, scipy.__version__
    except Exception:
        pass
    return d


def available_gb() -> float:
    """Free plus reclaimable memory in GB."""
    if platform.system() == "Darwin":
        try:
            out = subprocess.run(["vm_stat"], capture_output=True, text=True, timeout=5).stdout
            page = 16384
            first = out.splitlines()[0]
            if "page size of" in first:
                page = int(first.split("page size of")[1].split()[0])
            pages = 0
            for ln in out.splitlines()[1:]:
                key, _, val = ln.partition(":")
                if key.strip() in ("Pages free", "Pages inactive", "Pages speculative", "Pages purgeable"):
                    pages += int(val.strip().rstrip("."))
            return pages * page / 2 ** 30
        except Exception:
            return float("inf")
    try:
        for ln in open("/proc/meminfo"):
            if ln.startswith("MemAvailable"):
                return int(ln.split()[1]) / 2 ** 20
    except OSError:
        pass
    return float("inf")


def free_disk_gb(path: Path) -> float:
    try:
        st = os.statvfs(path)
        return st.f_bavail * st.f_frsize / 2 ** 30
    except OSError:
        return float("inf")


def paths(runs: Path, job: dict):
    base = runs / job["block"] / f"rep_{job['rep']:03d}"
    return base / f"{job['cell']}.json", base / f"{job['cell']}.log", base / f"{job['cell']}.progress.json"


def settled(runs: Path, job: dict, retry_failed: bool) -> bool:
    """A job that needs no run: finished, or failed without --retry-failed.
    A job skipped for the deadline runs again in a new campaign."""
    rec = read_json(paths(runs, job)[0])
    if not rec:
        return False
    return rec.get("status") == "ok" or (rec.get("status") == "failed" and not retry_failed)


def license_info() -> dict:
    """Kind of Gurobi license file: 'wls' (Web License Service), 'file'
    (named-user or node license) or 'missing'. The file does not state how
    many sessions may run at once; probe_sessions measures that."""
    cands = [os.environ.get("GRB_LICENSE_FILE"), str(Path.home() / "gurobi.lic"),
             "/Library/gurobi/gurobi.lic", "/opt/gurobi/gurobi.lic"]
    for c in cands:
        if c and Path(c).is_file():
            try:
                text = Path(c).read_text(errors="replace").upper()
            except OSError:
                continue
            kind = "wls" if "WLSACCESSID" in text else "file"
            return dict(path=c, kind=kind)
    return dict(path=None, kind="missing")


def _probe(i, outq):
    try:
        import gurobipy as gp
        env = gp.Env(empty=True)
        env.setParam("OutputFlag", 0)
        env.start()
        m = gp.Model(env=env)
        x = m.addVars(2600, vtype="B")
        m.setObjective(x.sum())
        m.addConstr(x.sum() >= 1)
        m.optimize()
        ok = m.Status == 2
        time.sleep(3.0)                     # hold the session while the others start
        env.dispose()
        outq.put((i, ok, None))
    except Exception as ex:
        outq.put((i, False, repr(ex)))


def probe_sessions(n: int) -> dict:
    """Start n Gurobi environments at once, each solving a 2,600-variable
    model, and count how many succeed: the number of workers the license
    supports. Each started environment is one license session."""
    ctx = mp.get_context("spawn")
    outq = ctx.Queue()
    procs = [ctx.Process(target=_probe, args=(i, outq)) for i in range(n)]
    for p in procs:
        p.start()
    res = []
    for _ in procs:
        try:
            res.append(outq.get(timeout=120))
        except queue_mod.Empty:
            break
    for p in procs:
        p.join(timeout=10)
    errors = sorted({e for _, ok, e in res if e})
    return dict(asked=n, succeeded=sum(1 for _, ok, _ in res if ok), errors=errors)


def default_workers(m: dict, threads: int) -> int:
    cores = m.get("physical_cpus") or os.cpu_count() or 1
    return max(1, (cores - 2) // threads) if cores > 4 else max(1, cores // threads)


# ------------------------------------------------------------------ worker
def _worker(wid: int, cfg: dict, runs: str, inbox, outbox, fake: bool):
    sys.path.insert(0, str(HERE))
    signal.signal(signal.SIGINT, signal.SIG_IGN)
    runs = Path(runs)
    import kp_blocks as kb
    env = None
    if not fake:
        try:
            import kp_solve as ks
            env = ks.make_env(cfg["threads"])
        except Exception as ex:
            outbox.put(dict(kind="env_error", wid=wid, error=repr(ex)))
            return
    run = kb.fake_run_job if fake else kb.run_job
    outbox.put(dict(kind="ready", wid=wid, pid=os.getpid()))
    while True:
        job = inbox.get()
        if job is None:
            break
        res_path, log_path, prog_path = paths(runs, job)
        res_path.parent.mkdir(parents=True, exist_ok=True)
        t0 = time.time()
        rec = dict(job, started=now(), worker=wid, pid=os.getpid(), host=socket.gethostname())
        try:
            rec["result"] = run(job, cfg, env, log_file=str(log_path), progress=str(prog_path))
            rec["status"] = "ok"
        except Exception as ex:
            rec["status"] = "failed"
            rec["error"] = repr(ex)
            rec["trace"] = traceback.format_exc()
            rec["error_kind"] = "license" if "licen" in repr(ex).lower() else (
                "memory" if "memory" in repr(ex).lower() else "error")
        rec["wall"] = time.time() - t0
        rec["finished"] = now()
        write_json(res_path, rec)
        try:
            prog_path.unlink()
        except OSError:
            pass
        outbox.put(dict(kind="done", wid=wid, id=job["id"], status=rec["status"], wall=rec["wall"],
                        error_kind=rec.get("error_kind")))


# --------------------------------------------------------------- scheduler
class Campaign:
    def __init__(self, cfg, queue, runs: Path, workers: int, mem_gb: float, retry_failed: bool, fake: bool,
                 deadline: float = float("inf")):
        self.cfg, self.queue, self.runs = cfg, queue, runs
        self.deadline = deadline
        self.skipped = 0
        self.J, self.mem_gb, self.retry_failed, self.fake = workers, mem_gb, retry_failed, fake
        self.ctx = mp.get_context("spawn")
        self.outbox = self.ctx.Queue()
        self.workers = {}            # wid -> dict(proc, inbox, job, started, ready)
        self.attempts = read_json(runs / "_attempts.json", {}) or {}
        self.events, self.issues = [], []
        self.stop_flag = False
        self.counts = dict(ok=0, failed=0)
        self.env_errors = 0
        self.next_wid = 0

    def skip(self, job, left):
        """Record a job that cannot finish before the deadline; a failed record is kept."""
        res_path = paths(self.runs, job)[0]
        prev = read_json(res_path, {}) or {}
        if prev.get("status") != "failed":
            write_json(res_path, dict(job, status="skipped", error_kind="deadline",
                                      error=f"{max(0.0, left) / 60:.0f} min left before the campaign deadline",
                                      finished=now()))
        self.skipped += 1

    def log(self, msg):
        line = f"{now()} {msg}"
        print(line, flush=True)
        self.events = (self.events + [line])[-30:]
        with open(self.runs / "scheduler.log", "a") as f:
            f.write(line + "\n")

    def spawn(self):
        wid = self.next_wid
        self.next_wid += 1
        inbox = self.ctx.Queue()
        p = self.ctx.Process(target=_worker, args=(wid, self.cfg, str(self.runs), inbox, self.outbox, self.fake),
                             daemon=True)
        p.start()
        self.workers[wid] = dict(proc=p, inbox=inbox, job=None, started=None, ready=False)

    def status(self, pending):
        running = [dict(id=w["job"]["id"], set=w["job"]["set"], worker=wid, pid=w["proc"].pid, started=w["started"])
                   for wid, w in self.workers.items() if w["job"]]
        write_json(self.runs / "status.json", dict(
            updated=now(), scheduler_pid=os.getpid(), host=socket.gethostname(), workers=len(self.workers),
            workers_target=self.J, running=running, pending=len(pending), ok=self.counts["ok"],
            failed=self.counts["failed"], skipped=self.skipped, total=len(self.queue),
            deadline=dt.datetime.fromtimestamp(self.deadline).isoformat(timespec="seconds") if self.deadline < float("inf") else None,
            hours_left=round((self.deadline - time.time()) / 3600, 2) if self.deadline < float("inf") else None,
            time_limit=self.cfg.get("time_limit"), available_gb=round(available_gb(), 1),
            mem_gb_per_job=self.mem_gb, free_disk_gb=round(free_disk_gb(self.runs), 1),
            issues=self.issues[-20:], events=self.events[-15:], stopping=self.stop_flag))

    def run(self):
        pending = [j for j in self.queue if not settled(self.runs, j, self.retry_failed)]
        done_before = len(self.queue) - len(pending)
        self.log(f"start: {len(self.queue)} jobs, {done_before} settled, {len(pending)} to run, "
                 f"{self.J} workers x {self.cfg['threads']} threads, {self.mem_gb:.1f} GB per job")
        if pending and time.time() >= self.deadline:     # relaunch after the deadline: close the campaign
            for job in pending:
                self.skip(job, self.deadline - time.time())
            self.log(f"deadline passed: {len(pending)} unfinished jobs recorded as skipped")
            pending = []
        for _ in range(min(self.J, len(pending))):
            self.spawn()
        last_status = 0.0
        while (pending or any(w["job"] for w in self.workers.values())) and not self.stop_flag:
            # messages from workers
            try:
                while True:
                    msg = self.outbox.get(timeout=1.0)
                    w = self.workers.get(msg["wid"])
                    if msg["kind"] == "ready" and w:
                        w["ready"] = True
                    elif msg["kind"] == "env_error":
                        self.env_errors += 1
                        self.issues.append(dict(kind="license", time=now(), detail=msg["error"]))
                        self.log(f"worker {msg['wid']} could not start Gurobi: {msg['error']}")
                        if w:
                            w["proc"].join(timeout=5)
                            self.workers.pop(msg["wid"], None)
                        self.J = max(1, len(self.workers)) if self.workers else 0
                    elif msg["kind"] == "done" and w:
                        self.counts[msg["status"]] = self.counts.get(msg["status"], 0) + 1
                        self.log(f"{msg['status']} {msg['id']} in {msg['wall']:.0f}s")
                        if msg["status"] != "ok":
                            self.issues.append(dict(kind=msg.get("error_kind") or "error", time=now(), job=msg["id"]))
                        w["job"], w["started"] = None, None
            except queue_mod.Empty:
                pass
            # dead workers
            for wid, w in list(self.workers.items()):
                if w["proc"].exitcode is not None:
                    job = w["job"]
                    self.workers.pop(wid)
                    if job:
                        n = self.attempts.get(job["id"], 0) + 1
                        self.attempts[job["id"]] = n
                        write_json(self.runs / "_attempts.json", self.attempts)
                        self.log(f"worker {wid} died (exit {w['proc'].exitcode}) during {job['id']}, attempt {n}")
                        if n < 2:
                            pending.insert(0, job)
                        else:
                            res_path = paths(self.runs, job)[0]
                            write_json(res_path, dict(job, status="failed", error_kind="worker_died",
                                                      error=f"worker died {n} times (exit {w['proc'].exitcode})",
                                                      finished=now()))
                            self.counts["failed"] += 1
                            self.issues.append(dict(kind="worker_died", time=now(), job=job["id"]))
                    if pending and self.env_errors == 0:
                        self.spawn()
            if not self.workers and pending:
                self.log("no worker can start; stopping")
                break
            # dispatch
            for wid, w in self.workers.items():
                if not pending:
                    break
                if w["ready"] and w["job"] is None:
                    if available_gb() < self.mem_gb and any(x["job"] for x in self.workers.values()):
                        break                     # wait for memory while other jobs run
                    job = pending.pop(0)
                    if settled(self.runs, job, self.retry_failed):
                        continue
                    import kp_blocks as kb
                    left = self.deadline - time.time()
                    need = self.cfg["job_overhead"] + job["n_ev"] * min(30.0, self.cfg["ev_time_limit"]) + job["n_mip"] * 60.0
                    if left < need:
                        self.skip(job, left)
                        continue
                    job = dict(job, deadline=time.time() + min(left - 30.0, kb.job_seconds(job, self.cfg)))
                    w["job"], w["started"] = job, now()
                    w["inbox"].put(job)
            if time.time() - last_status >= 10:
                self.status(pending)
                last_status = time.time()
        self.status(pending)
        for w in self.workers.values():
            try:
                w["inbox"].put(None)
            except Exception:
                pass
        deadline = time.time() + (5 if not self.stop_flag else 15)
        for w in self.workers.values():
            w["proc"].join(timeout=max(0.1, deadline - time.time()))
            if w["proc"].is_alive():
                w["proc"].terminate()
        return pending


def _write_plan(runs: Path, cfg: dict, queue: list):
    plan = dict(written=now(), total=len(queue),
                sets={}, jobs=[{k: j[k] for k in ("id", "set", "block", "rep", "cell", "wc")} for j in queue])
    for j in queue:
        plan["sets"].setdefault(j["set"], []).append(j["id"])
    write_json(runs / "plan.json", plan)


def cmd_plan(args, cfg, queue, J, m, lic, budget_info):
    cores = m.get("physical_cpus") or os.cpu_count()
    by = {}
    for j in queue:
        d = by.setdefault(j["block"], dict(jobs=0, sets=set(), wc=0.0))
        d["jobs"] += 1
        d["sets"].add(j["set"])
        d["wc"] += j["wc"]
    print(f"machine: {m.get('chip')}, {cores} physical cores ({m.get('performance_cpus')} performance), "
          f"{m.get('memory_gb')} GB, Gurobi {m.get('gurobi')}")
    print(f"license: {lic['kind']} ({lic['path']})")
    print(f"{J} workers x {cfg['threads']} threads; budget {cfg['campaign_hours']:.0f} h; time limit "
          f"{cfg['time_limit']:.0f} s per solve; stop at gap {cfg['mip_gap']:.0%} (X7: {cfg['x7_mip_gap']:.0%})")
    if budget_info.get("cuts"):
        from collections import Counter
        print(f"replications cut to fit the budget: {dict(Counter(budget_info['cuts']))}; now {budget_info['reps']}")
    tot = 0.0
    for b, d in by.items():
        tot += d["wc"]
        print(f"  {b}: {d['jobs']} jobs in {len(d['sets'])} sets, worst case {d['wc'] / 3600:.1f} job-hours")
    print(f"total: {len(queue)} jobs, worst case {tot / 3600:.1f} job-hours, about "
          f"{tot / 3600 / J / cfg['efficiency']:.1f} h of wall time with {J} workers if every solve reaches its "
          f"time limit; solves that prove the gap earlier finish sooner")
    return dict(workers=J, job_hours=tot / 3600)


def main(argv=None):
    sys.path.insert(0, str(HERE))
    import kp_blocks as kb
    ap = argparse.ArgumentParser()
    ap.add_argument("command", choices=["plan", "launch", "preflight", "status", "stop"])
    ap.add_argument("--workers", type=int, default=None)
    ap.add_argument("--threads", type=int, default=None)
    ap.add_argument("--blocks", default=None, help="comma-separated, for example X1,X7")
    ap.add_argument("--retry-failed", action="store_true")
    ap.add_argument("--runs", default=None)
    ap.add_argument("--config", default=None, help="JSON file of overrides of kp_blocks.DEFAULTS")
    ap.add_argument("--mem-gb", type=float, default=None, help="memory allowance per job")
    ap.add_argument("--fake", action="store_true", help="no solver: test the scheduler")
    ap.add_argument("--hours", type=float, default=None, help="campaign budget in hours (default 48)")
    ap.add_argument("--new-campaign", action="store_true", help="start a new budget instead of resuming the deadline")
    ap.add_argument("--time-limit", type=float, default=None, help="fixed seconds per solve instead of the budget")
    ap.add_argument("--detach", action="store_true",
                    help="launch in its own session under caffeinate, log to runs/scheduler.out, and return")
    args = ap.parse_args(argv)
    runs = Path(args.runs) if args.runs else (RUNS / "_preflight" if args.command == "preflight" else RUNS)
    if args.detach and args.command in ("launch", "preflight"):
        rest = [a for a in (sys.argv[1:] if argv is None else list(argv)) if a != "--detach"]
        cmd = [sys.executable, str(Path(__file__).resolve()), *rest]
        if shutil.which("caffeinate"):
            cmd = ["caffeinate", "-ims"] + cmd
        runs.mkdir(parents=True, exist_ok=True)
        with open(runs / "scheduler.out", "a") as out:
            p = subprocess.Popen(cmd, cwd=str(HERE), stdout=out, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                                 start_new_session=True)
        print(f"started the scheduler in its own session: pid {p.pid}; log {runs / 'scheduler.out'}")
        return 0
    if args.command == "status":
        print(json.dumps(read_json(runs / "status.json", {}), indent=1))
        return 0
    if args.command == "stop":
        write_json(runs / "STOPPED.json", dict(time=now(), by=os.getpid()))    # the monitor does not relaunch
        lock = read_json(runs / "run.lock", {})
        if lock.get("pid") and pid_alive(lock["pid"]):
            os.kill(int(lock["pid"]), signal.SIGTERM)
            print(f"sent SIGTERM to scheduler {lock['pid']}")
        else:
            print("no scheduler is running")
        return 0
    over = read_json(Path(args.config), {}) if args.config else {}
    cfg = kb.preflight_config() if args.command == "preflight" else kb.config(over)
    if args.threads:
        cfg["threads"] = args.threads
    if args.hours:
        cfg["campaign_hours"] = args.hours
    blocks = args.blocks.split(",") if args.blocks else None
    m = machine()
    lic = license_info()
    J = args.workers or default_workers(m, cfg["threads"])
    if args.command == "preflight":
        J = min(J, 2)
    old = read_json(runs / "campaign.json", {}) or {}
    # A relaunch keeps the first launch's deadline and plan, also after the deadline has passed: the
    # scheduler then records every unfinished job as skipped and closes the campaign.
    resume = bool(old.get("deadline")) and not args.new_campaign and args.command == "launch" \
        and old.get("blocks") == (blocks or cfg["blocks"])
    if resume:
        cfg = old["config"]
        deadline = float(old["deadline"])
        budget_info = old.get("budget", {})
    else:
        if args.time_limit:
            cfg["time_limit"] = args.time_limit
            budget_info = dict(time_limit=args.time_limit, fixed=True)
        elif args.command == "preflight" or "time_limit" in over:
            budget_info = dict(time_limit=cfg["time_limit"], fixed=True)
        else:
            budget_info = kb.budget(cfg, J, blocks)
            cfg = budget_info.pop("config")
        deadline = time.time() + cfg["campaign_hours"] * 3600.0
    queue = kb.build_queue(cfg, blocks)
    if args.command == "plan":
        cmd_plan(args, cfg, queue, J, m, lic, budget_info)
        return 0
    runs.mkdir(parents=True, exist_ok=True)
    lock_path = runs / "run.lock"
    lock = read_json(lock_path, {})
    if lock.get("pid") and pid_alive(lock["pid"]) and lock.get("host") == socket.gethostname():
        print(f"A campaign is already running (pid {lock['pid']}, started {lock.get('started')}). "
              f"Stop it with: python {Path(__file__).name} stop")
        return 3
    mem_total = m.get("memory_gb") or 16.0
    mem_gb = args.mem_gb or max(1.0, 0.8 * mem_total / J)
    if cfg.get("soft_mem_gb") is None:
        cfg["soft_mem_gb"] = round(mem_gb, 1)
    write_json(lock_path, dict(pid=os.getpid(), host=socket.gethostname(), started=now(), command=args.command))
    try:
        (runs / "STOPPED.json").unlink()
    except OSError:
        pass
    first = old.get("first_started") if resume else now()
    write_json(runs / "campaign.json", dict(
        first_started=first, started=now(), resumed=(old.get("resumed", []) if resume else []) + [now()],
        deadline=deadline, deadline_iso=dt.datetime.fromtimestamp(deadline).isoformat(timespec="seconds"),
        command=args.command, argv=sys.argv, python=sys.executable, workers=J, threads=cfg["threads"],
        mem_gb_per_job=mem_gb,
        blocks=blocks or cfg["blocks"], machine=m, license=lic, budget=budget_info, config=cfg))
    _write_plan(runs, cfg, queue)
    try:
        (runs / "CAMPAIGN_DONE.json").unlink()
    except OSError:
        pass
    camp = Campaign(cfg, queue, runs, J, mem_gb, args.retry_failed, args.fake, deadline)

    def handler(signum, frame):
        camp.stop_flag = True
        camp.log(f"signal {signum}: stopping after the running jobs are terminated")

    signal.signal(signal.SIGTERM, handler)
    signal.signal(signal.SIGINT, handler)
    try:
        left = camp.run()
        st = [(read_json(paths(runs, j)[0], {}) or {}).get("status") for j in queue]
        ok, failed, skipped = st.count("ok"), st.count("failed"), st.count("skipped")
        if not left and not camp.stop_flag:          # written before the lock goes, so no look sees a gap
            write_json(runs / "CAMPAIGN_DONE.json", dict(finished=now(), ok=ok, failed=failed, skipped=skipped,
                                                         total=len(queue), deadline=deadline))
    finally:
        try:
            lock_path.unlink()
        except OSError:
            pass
    camp.log(f"end: {ok} ok, {failed} failed, {skipped} skipped for the deadline, {len(left)} not run")
    return 0 if not left else 1


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writecode Implementation/kp_cuts.py
"""Closed-form recourse, Benders cuts and local search for the extended
formulation. NumPy only; no solver.

The solvers use the lower-bound form of the recourse. For scenario s it is

    min sum_i ell_i e_i
    s.t. e_i >= sum_m xi_im x_im - delta_i y_i                 (own error)
         e_i >= e_j + theta_ji - 1 - delta_i y_i               (handoff j -> i)
         e_i >= 0,

with theta_ji the share of the handoff's model pair that passes the error:
kappa_diff + (kappa_same - kappa_diff) z_ji when passing depends only on
whether the two models coincide ('z' mode), or sum over pairs of
kappa_mm' w_ji^mm' in general ('w' mode). A handoff whose every pair passes
has theta = 1. Because every loss weight is nonnegative and the systemic cap
only bounds sum_i e_i from above, the least solution of these rows is
optimal and equals the deck's exact Boolean recourse at every binary plan.
Its value is found in one pass in topological order, the attaining row of
each output names its parent, and the subtree weights of the resulting
forest are an optimal dual solution, as in the manuscript's Proposition 1.
"""
from __future__ import annotations

import time

import numpy as np

import kp_model as km


class CutEngine:
    """Scenario losses and optimality cuts of the extended recourse for a
    fixed workflow (every task used).

    mode 'z': first-stage pair variables z (nA,), z = 1 when both tasks of a
    handoff run on one model. mode 'w': pair variables w (nA, M, M)."""

    def __init__(self, inst: km.Instance, sc: km.Scenarios, weights=None, mode: str = "z"):
        self.inst, self.sc, self.mode = inst, sc, mode
        self.ell = inst.ell if weights is None else np.asarray(weights, float)
        self.xi_f = sc.xi.astype(np.float64)
        self.delta_f = sc.delta.astype(np.float64)
        q = sc.q
        U = sc.U
        self.always = U < q.min()                      # (S, nA): every pair passes
        self.live = U < q.max()                         # some pair passes
        if mode == "z":
            sd = km.same_diff(q)
            if sd is None:
                raise ValueError("mode 'z' needs pass probabilities that depend only on same or different models")
            qs, qd = sd
            self.k_same = (U < qs).astype(np.float64)
            self.k_diff = (U < qd).astype(np.float64)
        else:
            self.kap = U[:, :, None, None] < q[None, None, :, :]     # (S, nA, M, M)

    def theta(self, zv=None, wv=None) -> np.ndarray:
        """(S, nA) transmission share of each handoff for pair values zv or wv."""
        if self.mode == "z":
            return self.k_diff + (self.k_same - self.k_diff) * np.asarray(zv)[None, :]
        return np.einsum("skab,kab->sk", self.kap.astype(np.float64), np.asarray(wv))

    def evaluate(self, xv, yv, zv=None, wv=None):
        """Least solution e (S, n), loss (S,), parent (S, n) and dual weights.

        parent: -2 output not in the forest (e = 0), -1 own-error root,
        k >= 0 the handoff whose row attains e_i."""
        inst = self.inst
        S, n = self.sc.S, inst.n
        own = np.einsum("sim,im->si", self.xi_f, np.asarray(xv, float))
        dy = self.delta_f * np.asarray(yv, float)[None, :]
        th = self.theta(zv, wv)
        th = np.where(self.always, 1.0, th)
        e = np.zeros((S, n))
        parent = np.full((S, n), -2, dtype=np.int64)
        for i in range(n):
            best = own[:, i] - dy[:, i]
            arg = np.full(S, -1, dtype=np.int64)
            for k in inst.in_arcs[i]:
                j = inst.arcs[k, 0]
                cand = np.where(self.live[:, k], e[:, j] + th[:, k] - 1.0 - dy[:, i], -np.inf)
                upd = cand > best + 1e-12
                best = np.where(upd, cand, best)
                arg = np.where(upd, k, arg)
            pos = best > 1e-9
            e[:, i] = np.where(pos, best, 0.0)
            parent[:, i] = np.where(pos, arg, -2)
        Wt = np.where(parent != -2, self.ell[None, :], 0.0)
        for i in range(n - 1, -1, -1):
            for k in inst.in_arcs[i]:
                j = inst.arcs[k, 0]
                msk = parent[:, i] == k
                Wt[msk, j] += Wt[msk, i]
        return e, e @ self.ell, parent, Wt

    def cut(self, s: int, parent, Wt):
        """Cut of scenario s: L_s >= const + sum x_terms + sum y_terms + sum pair_terms.

        x_terms (i, m, coef); y_terms (i, coef); pair_terms (k, coef) in mode z
        or (k, m, m', coef) in mode w."""
        inst, sc = self.inst, self.sc
        xt, yt, pt, const = [], [], [], 0.0
        for i in np.nonzero(parent[s] != -2)[0]:
            w = float(Wt[s, i])
            if sc.delta[s, i]:
                yt.append((int(i), -w))
            k = int(parent[s, i])
            if k == -1:
                for m in np.nonzero(sc.xi[s, i])[0]:
                    xt.append((int(i), int(m), w))
            elif self.always[s, k]:
                pass                                   # theta = 1: no pair term
            elif self.mode == "z":
                const += w * (self.k_diff[s, k] - 1.0)
                coef = w * (self.k_same[s, k] - self.k_diff[s, k])
                if coef != 0.0:
                    pt.append((k, coef))
            else:
                const -= w
                for a_, b_ in zip(*np.nonzero(self.kap[s, k])):
                    pt.append((k, int(a_), int(b_), w))
        return xt, yt, pt, const

    def cut_value(self, s, parent, Wt, xv, yv, zv=None, wv=None) -> float:
        xt, yt, pt, const = self.cut(s, parent, Wt)
        val = const + sum(c * xv[i][m] for i, m, c in xt) + sum(c * yv[i] for i, c in yt)
        if self.mode == "z":
            val += sum(c * zv[k] for k, c in pt)
        else:
            val += sum(c * wv[k][a][b] for k, a, b, c in pt)
        return float(val)


def pair_values(inst: km.Instance, a, mode: str):
    """z (nA,) or w (nA, M, M) implied by a binary assignment a."""
    a = np.asarray(a, dtype=int)
    j, i = inst.arcs[:, 0], inst.arcs[:, 1]
    if mode == "z":
        return ((a[j] == a[i]) & (a[j] >= 0)).astype(float)
    w = np.zeros((inst.nA, inst.M, inst.M))
    ok = (a[j] >= 0) & (a[i] >= 0)
    w[np.nonzero(ok)[0], a[j][ok], a[i][ok]] = 1.0
    return w


def local_search(inst: km.Instance, sc: km.Scenarios, a0, y0, lam: float, alpha: float,
                 seconds: float = 30.0, cap_beta: float | None = None, seed: int = 0,
                 max_passes: int = 50):
    """First-improvement local search on a training sample, used to improve a
    warm start before the solver: change the model of one task, add a review,
    or move a review, keeping budget, hours and quotas. The set of used tasks
    stays fixed. With cap_beta the share of systemic failures above the cap is
    penalized, so a start that meets the cap is preferred.

    Returns (a, y, value, stats); value is the training objective."""
    rng = np.random.default_rng(seed)
    a = np.asarray(a0, dtype=int).copy()
    y = np.asarray(y0).astype(bool).copy()
    used = a >= 0

    def value(a_, y_):
        e = km.propagate(inst, a_, y_, sc)
        v = km.objective(inst, a_, y_, sc, lam, alpha, e=e)
        if cap_beta is not None:
            over = float(np.mean(e.sum(axis=1) >= inst.e_sys)) - cap_beta
            v += 1e3 * max(0.0, over)
        return v

    t0 = time.time()
    best = value(a, y)
    start_val = best
    moves = evals = passes = 0
    idx = np.nonzero(used)[0]
    while passes < max_passes and time.time() - t0 < seconds:
        passes += 1
        improved = False
        cand = [("model", int(i), int(m)) for i in idx for m in range(inst.M) if m != a[i]]
        cand += [("add", int(i), -1) for i in idx if not y[i]]
        cand += [("move", int(i), int(j)) for i in idx if y[i] for j in idx if not y[j]]
        rng.shuffle(cand)
        for kind, i, m in cand:
            if time.time() - t0 >= seconds:
                break
            a2, y2 = a, y
            if kind == "model":
                a2 = a.copy()
                a2[i] = m
            elif kind == "add":
                y2 = y.copy()
                y2[i] = True
            else:
                y2 = y.copy()
                y2[i], y2[m] = False, True
            if not km.feasible(inst, a2, y2):
                continue
            evals += 1
            v = value(a2, y2)
            if v < best - 1e-9:
                a, y, best = a2, y2, v
                moves += 1
                improved = True
        if not improved:
            break
    return a, y, best, dict(ls_start=start_val, ls_value=best, ls_moves=moves, ls_evals=evals,
                            ls_passes=passes, ls_seconds=time.time() - t0)


In [ ]:
%%writecode Implementation/kp_model.py
"""Extended model of the formulation deck: instances, delegation templates,
scenarios, error propagation and scores. NumPy and SciPy only; no solver.

The layered workflow and Gaussian copula use a common stream of random draws
so the baseline instances and scenarios match across plan comparisons.
Every extension (model-pair transmission, fallible review, loss weights,
provider quotas, Student t copula, posterior error rates, delegation
templates) draws from a second random stream and leaves those draws intact.
"""
from __future__ import annotations

import math
from dataclasses import dataclass, field

import numpy as np
from scipy.stats import norm
from scipy.stats import t as student_t

# Model pool of the manuscript: two frontier models from different providers
# (A, B), a mid-size model (C) and a small model (D).
MODEL_NAMES = ["A", "B", "C", "D"]
MODEL_P = np.array([0.010, 0.013, 0.025, 0.050])
MODEL_COST = np.array([10.0, 9.0, 3.0, 1.0])
# Provider of each model, used only by the provider quotas. Assumption of
# this study: A and C come from provider 0, B and D from provider 1.
MODEL_PROVIDER = np.array([0, 1, 0, 1])


@dataclass
class Instance:
    n: int
    M: int
    arcs: np.ndarray              # (nA, 2): tail j, head i; task i reads task j
    layer: np.ndarray             # depth of each task (stage of a layered workflow)
    sinks: np.ndarray             # deliverables, always used
    ell: np.ndarray               # loss weight of each output
    c: np.ndarray                 # (n, M) cost of task i on model m
    p: np.ndarray                 # (n, M) marginal silent-error probability
    h: np.ndarray                 # review hours of each output
    dbar: np.ndarray              # probability that a review catches the error
    B: float                      # spend budget
    H: float                      # review-hours budget
    Q: np.ndarray                 # spend quota per provider (inf when absent)
    e_sys: int                    # wrong outputs that make a run a systemic failure
    seed: int
    kind: str = "layered"         # layered, tree or random
    and_groups: dict = field(default_factory=dict)   # task -> sub-tasks, all used
    or_groups: dict = field(default_factory=dict)    # task -> alternatives, one used
    in_arcs: list = field(default_factory=list)

    @property
    def nA(self) -> int:
        return len(self.arcs)

    @property
    def fixed(self) -> bool:
        """True when every task is used (no delegation alternatives)."""
        return not self.and_groups and not self.or_groups

    def default_use(self) -> np.ndarray:
        """Tasks used when every OR group takes its first alternative."""
        if self.fixed:
            return np.ones(self.n, dtype=bool)
        use = np.zeros(self.n, dtype=bool)
        stack = [int(r) for r in self.sinks]
        while stack:
            i = stack.pop()
            use[i] = True
            if i in self.and_groups:
                stack += list(self.and_groups[i])
            if i in self.or_groups:
                stack.append(int(self.or_groups[i][0]))
        return use


def _in_arcs(n, arcs):
    out = [[] for _ in range(n)]
    for k, (j, i) in enumerate(arcs):
        out[int(i)].append(k)
    return out


def make_instance(L: int, W: int, fan: float, beta: float, tau: float, seed: int,
                  p_scale: float = 1.0, ell_mode: str = "all", dbar: float = 1.0,
                  quota_share: float | None = None, e_sys_share: float = 0.25) -> Instance:
    """Layered workflow of the manuscript: L stages of W tasks; each task reads
    1 to 3 outputs of the previous stage (mean `fan`); the last stage holds the
    deliverables. ell_mode 'all' weighs every output 1; 'deliverables' weighs
    only the last stage. quota_share caps each provider's spend at that share
    of B (None: no quota)."""
    rng = np.random.default_rng(seed)
    n = L * W
    M = len(MODEL_NAMES)
    layer = np.repeat(np.arange(L), W)
    arcs = set()
    for l in range(1, L):
        prev = np.arange((l - 1) * W, l * W)
        cur = np.arange(l * W, (l + 1) * W)
        for i in cur:
            k = 1 + rng.binomial(2, min(1.0, max(0.0, (fan - 1) / 2)))
            k = min(k, W)
            for j in rng.choice(prev, size=k, replace=False):
                arcs.add((int(j), int(i)))
        tails = {j for (j, i) in arcs if layer[i] == l}
        for j in prev:
            if int(j) not in tails:
                arcs.add((int(j), int(rng.choice(cur))))
    arcs = np.array(sorted(arcs, key=lambda a: (a[1], a[0])), dtype=int)
    sinks = np.arange((L - 1) * W, n)
    size = rng.uniform(0.5, 1.5, n)
    diff = np.clip(np.exp(rng.normal(0.0, 0.35, n)), 0.5, 2.0)
    c = size[:, None] * MODEL_COST[None, :]
    p = np.minimum(0.5, p_scale * diff[:, None] * MODEL_P[None, :])
    h = rng.uniform(1.0, 3.0, n)
    cmin, cmax = c.min(axis=1).sum(), c.max(axis=1).sum()
    B = cmin + beta * (cmax - cmin)
    H = tau * h.sum()
    ell = np.ones(n)
    if ell_mode == "deliverables":
        ell = np.zeros(n)
        ell[sinks] = 1.0
    Q = np.full(int(MODEL_PROVIDER.max()) + 1, np.inf)
    if quota_share is not None:
        Q[:] = quota_share * B
    return Instance(n=n, M=M, arcs=arcs, layer=layer, sinks=sinks, ell=ell, c=c, p=p, h=h,
                    dbar=np.full(n, float(dbar)), B=float(B), H=float(H), Q=Q,
                    e_sys=int(math.ceil(e_sys_share * n - 1e-9)), seed=seed,
                    in_arcs=_in_arcs(n, arcs))


def make_template(kind: str, seed: int, beta: float, tau: float, p_scale: float = 1.0,
                  dbar: float = 1.0, e_sys_share: float = 0.25) -> Instance:
    """Delegation template after Baqueta and Tacla (2026): a tree of task
    occurrences rooted at one deliverable, which delegates to several
    sub-goals. Below the root, a goal has an AND group (every sub-task used)
    or, with probability 0.5, an OR group with two alternatives: a direct
    task, harder and cheaper, or a merge task with an AND group of sub-tasks.
    Outputs flow from sub-tasks to the task that delegated them.

    kind 'tree': root with 4 sub-goals, groups of 3, depth 3. kind 'random':
    root with 3 to 5 sub-goals, groups of 2 to 4, depth up to 4, and a leaf
    with probability 0.2 times the depth from depth 2."""
    rng = np.random.default_rng([seed, 31])
    nodes = []                       # [depth, kind, children]; kind leaf, and, or, direct
    if kind == "tree":
        depth_max, p_or, root_b = 3, 0.5, 4
    elif kind == "random":
        depth_max, p_or, root_b = 4, 0.5, int(rng.integers(3, 6))
    else:
        raise ValueError(kind)

    def branching():
        return 3 if kind == "tree" else int(rng.integers(2, 5))

    def build(depth):
        idx = len(nodes)
        nodes.append([depth, "leaf", []])
        if depth >= depth_max or (kind == "random" and depth >= 2 and rng.random() < 0.2 * depth):
            return idx
        if depth == 0:
            nodes[idx][1], nodes[idx][2] = "and", [build(1) for _ in range(root_b)]
        elif rng.random() < p_or:
            direct = len(nodes)
            nodes.append([depth + 1, "direct", []])
            merge = len(nodes)
            nodes.append([depth + 1, "and", []])
            nodes[merge][2] = [build(depth + 1) for _ in range(branching())]
            nodes[idx][1], nodes[idx][2] = "or", [direct, merge]
        else:
            nodes[idx][1], nodes[idx][2] = "and", [build(depth + 1) for _ in range(branching())]
        return idx

    build(0)
    # post-order numbering: sub-tasks before the task that reads them
    order, seen = [], set()

    def visit(k):
        for ch in nodes[k][2]:
            visit(ch)
        order.append(k)
        seen.add(k)

    visit(0)
    new = {old: i for i, old in enumerate(order)}
    n = len(order)
    M = len(MODEL_NAMES)
    arcs, and_groups, or_groups = [], {}, {}
    depth = np.zeros(n, dtype=int)
    kinds = [""] * n
    for old, i in new.items():
        d, kd, ch = nodes[old]
        depth[i], kinds[i] = d, kd
        kids = [new[c] for c in ch]
        for j in kids:
            arcs.append((j, i))
        if kd == "and":
            and_groups[i] = kids
        elif kd == "or":
            or_groups[i] = kids
    arcs = np.array(sorted(arcs, key=lambda a: (a[1], a[0])), dtype=int).reshape(-1, 2)
    size = rng.uniform(0.5, 1.5, n)
    diff = np.clip(np.exp(rng.normal(0.0, 0.35, n)), 0.5, 2.0)
    # A direct task does the work of the merge it replaces: harder, and cheaper
    # than the merge with its sub-tasks.
    for i, kd in enumerate(kinds):
        if kd == "direct":
            diff[i] = min(4.0, 2.5 * diff[i])
            size[i] = 1.5 * size[i]
    c = size[:, None] * MODEL_COST[None, :]
    p = np.minimum(0.5, p_scale * diff[:, None] * MODEL_P[None, :])
    h = rng.uniform(1.0, 3.0, n)
    sinks = np.array([n - 1])
    inst = Instance(n=n, M=M, arcs=arcs, layer=depth.max() - depth, sinks=sinks, ell=np.ones(n),
                    c=c, p=p, h=h, dbar=np.full(n, float(dbar)), B=0.0, H=0.0,
                    Q=np.full(int(MODEL_PROVIDER.max()) + 1, np.inf), e_sys=1, seed=seed,
                    kind=kind, and_groups=and_groups, or_groups=or_groups,
                    in_arcs=_in_arcs(n, arcs))
    # Budgets relative to the deepest decomposition (every OR group delegates),
    # so that the deepest one is affordable at share beta and shallower ones cost less.
    deep = np.zeros(n, dtype=bool)
    stack = [n - 1]
    while stack:
        i = stack.pop()
        deep[i] = True
        if i in and_groups:
            stack += and_groups[i]
        if i in or_groups:
            stack.append(or_groups[i][-1])
    cmin, cmax = c[deep].min(axis=1).sum(), c[deep].max(axis=1).sum()
    inst.B = float(cmin + beta * (cmax - cmin))
    inst.H = float(tau * h[deep].sum())
    inst.e_sys = int(math.ceil(e_sys_share * deep.sum() - 1e-9))
    return inst


def q_matrix(q_same: float, q_diff: float, M: int = len(MODEL_NAMES)) -> np.ndarray:
    """Pass probability of a handoff from model m to model m': q_same on the
    diagonal, q_diff elsewhere."""
    q = np.full((M, M), float(q_diff))
    np.fill_diagonal(q, float(q_same))
    return q


def same_diff(q: np.ndarray):
    """(q_same, q_diff) when q has that structure, else None."""
    d = np.diag(q)
    off = q[~np.eye(len(q), dtype=bool)]
    if np.allclose(d, d[0]) and np.allclose(off, off[0]):
        return float(d[0]), float(off[0])
    return None


@dataclass
class Scenarios:
    xi: np.ndarray        # (S, n, M) bool: model m errs silently on task i
    U: np.ndarray         # (S, nA) uniform draw of each handoff
    delta: np.ndarray     # (S, n) bool: a review of output i catches its error
    q: np.ndarray         # (M, M) pass probabilities of this sample
    meta: dict = field(default_factory=dict)

    @property
    def S(self) -> int:
        return self.xi.shape[0]

    def kappa(self, k: int, m: int, mm: int) -> np.ndarray:
        """(S,) bool: handoff k passes an upstream error from model m to model mm."""
        return self.U[:, k] < self.q[m, mm]


def sample_scenarios(inst: Instance, S: int, rho: float, gamma: float, q: np.ndarray, seed: int,
                     copula: str = "gaussian", nu: float = 4.0, review_tie: float | None = None,
                     posterior_k: float | None = None) -> Scenarios:
    """One-factor copula per model. rho is the latent correlation of one
    model's errors across tasks; gamma is the share of that model factor common
    to all models; marginal rates equal inst.p. A handoff passes an upstream
    error when its uniform draw is below q[m, m'] for the models of its two
    tasks, so all model pairs share common random numbers.

    copula 't' divides each scenario's latent variables by sqrt(W/nu), W
    chi-square with nu degrees, which adds tail dependence across all tasks
    and models. review_tie None draws review success independently with
    probability inst.dbar; a value r in [0, 1] draws a miss when
    sqrt(r) G + sqrt(1-r) eta falls below the quantile of 1 - dbar, with G
    the common factor of the errors, so reviews miss more often in scenarios
    where every model errs more. posterior_k draws each scenario's rates from
    Beta(k p, k (1 - p)), as from k logged outcomes."""
    rng = np.random.default_rng(seed)                 # the manuscript's stream
    rng2 = np.random.default_rng([seed, 7])           # extensions only
    n, M = inst.n, inst.M
    G = rng.standard_normal((S, 1))
    Fm = np.sqrt(gamma) * G + np.sqrt(1.0 - gamma) * rng.standard_normal((S, M))
    scale = None
    if copula == "t":
        scale = 1.0 / np.sqrt(rng2.chisquare(nu, size=S) / nu)
    elif copula != "gaussian":
        raise ValueError(copula)
    if posterior_k:
        k = float(posterior_k)
        a_ = np.clip(k * inst.p, 1e-6, None)
        b_ = np.clip(k * (1.0 - inst.p), 1e-6, None)
        p_s = rng2.beta(a_[None], b_[None], size=(S, n, M))
    else:
        p_s = None
    ppf = (lambda x: student_t.ppf(x, nu)) if copula == "t" else norm.ppf
    thr = ppf(inst.p)
    xi = np.empty((S, n, M), dtype=bool)
    chunk = max(1, 2_000_000 // (n * M))
    for s0 in range(0, S, chunk):
        s1 = min(S, s0 + chunk)
        eps = rng.standard_normal((s1 - s0, n, M))
        Z = np.sqrt(rho) * Fm[s0:s1, None, :] + np.sqrt(1.0 - rho) * eps
        if scale is not None:
            Z = Z * scale[s0:s1, None, None]
        th = ppf(p_s[s0:s1]) if p_s is not None else thr[None, :, :]
        xi[s0:s1] = Z < th
    U = rng.random((S, inst.nA))
    if review_tie is None and np.all(inst.dbar >= 1.0):
        delta = np.ones((S, n), dtype=bool)
    elif review_tie is None:
        delta = rng2.random((S, n)) < inst.dbar[None, :]
    else:
        r = float(review_tie)
        lat = np.sqrt(r) * G + np.sqrt(1.0 - r) * rng2.standard_normal((S, n))
        delta = lat >= norm.ppf(np.clip(1.0 - inst.dbar, 1e-12, 1 - 1e-12))[None, :]
    meta = dict(S=S, rho=rho, gamma=gamma, seed=seed, copula=copula, nu=nu,
                review_tie=review_tie, posterior_k=posterior_k)
    return Scenarios(xi=xi, U=U.astype(np.float64), delta=delta, q=np.asarray(q, float), meta=meta)


def propagate(inst: Instance, a, y, sc: Scenarios) -> np.ndarray:
    """(S, n) bool: outputs released wrong under assignment a (n,), with -1
    for an unused task, and reviews y (n,)."""
    a = np.asarray(a, dtype=int)
    y = np.asarray(y).astype(bool)
    S, n = sc.S, inst.n
    e = np.zeros((S, n), dtype=bool)
    for i in range(n):                                # index order is topological
        if a[i] < 0:
            continue
        val = sc.xi[:, i, a[i]].copy()
        for k in inst.in_arcs[i]:
            j = inst.arcs[k, 0]
            if a[j] < 0:
                continue
            val |= e[:, j] & (sc.U[:, k] < sc.q[a[j], a[i]])
        if y[i]:
            val &= ~sc.delta[:, i]
        e[:, i] = val
    return e


def cvar(loss: np.ndarray, alpha: float) -> float:
    """Rockafellar and Uryasev CVaR of an equally weighted sample."""
    S = len(loss)
    srt = np.sort(loss)[::-1]
    k = (1.0 - alpha) * S
    full = int(np.floor(k + 1e-9))
    tot = srt[:full].sum()
    if full < S and k - full > 1e-9:
        tot += (k - full) * srt[full]
    return float(tot / k)


def objective(inst, a, y, sc, lam, alpha, e=None) -> float:
    e = propagate(inst, a, y, sc) if e is None else e
    loss = e @ inst.ell
    return (1 - lam) * float(loss.mean()) + (lam * cvar(loss, alpha) if lam > 0 else 0.0)


def metrics(inst: Instance, a, y, sc: Scenarios, lam: float, alpha: float) -> dict:
    a = np.asarray(a, dtype=int)
    y = np.asarray(y).astype(bool)
    e = propagate(inst, a, y, sc)
    loss = e @ inst.ell
    count = e.sum(axis=1)
    used = a >= 0
    mean, cv = float(loss.mean()), cvar(loss, alpha)
    share = np.bincount(a[used], minlength=inst.M) / max(1, used.sum())
    both = used[inst.arcs[:, 0]] & used[inst.arcs[:, 1]] if inst.nA else np.zeros(0, bool)
    same = (a[inst.arcs[both, 0]] == a[inst.arcs[both, 1]]) if both.any() else np.zeros(0, bool)
    hrs = float(inst.h[y].sum())
    last = inst.layer == inst.layer.max()
    task_cost = np.where(used, inst.c[np.arange(inst.n), np.maximum(a, 0)], 0.0)
    prov = MODEL_PROVIDER[np.maximum(a, 0)]
    spend_k = [float(task_cost[used & (prov == k)].sum()) for k in range(len(inst.Q))]
    out = dict(
        exp_loss=mean, cvar=cv, risk_obj=(1 - lam) * mean + lam * cv,
        p_systemic=float(np.mean(count >= inst.e_sys)),
        p_clean=float(np.mean(count == 0)),
        wrong_per_run=float(count.mean()),
        p_any_deliverable=float(np.mean(e[:, inst.sinks].any(axis=1))),
        spend=float(inst.c[np.nonzero(used)[0], a[used]].sum()),
        review_hours=hrs, n_reviews=int(y.sum()), n_used=int(used.sum()),
        n_used_handoffs=int(both.sum()),
        eff_models=float(1.0 / np.sum(share ** 2)) if used.any() else 0.0,
        same_model_arc_share=float(same.mean()) if len(same) else 0.0,
        review_share_last_stage=float(y[last].sum() / y.sum()) if y.any() else 0.0,
        mean_review_depth=float(inst.layer[y].mean() / max(1, inst.layer.max())) if y.any() else -1.0,
        spend_provider=spend_k,
    )
    for m, nm in enumerate(MODEL_NAMES):
        out[f"share_{nm}"] = float(share[m])
    return out


def feasible(inst: Instance, a, y, tol: float = 1e-6) -> bool:
    """Budget, review hours, quotas and delegation groups of a plan."""
    a = np.asarray(a, dtype=int)
    y = np.asarray(y).astype(bool)
    used = a >= 0
    if np.any(y & ~used):
        return False
    if inst.c[np.nonzero(used)[0], a[used]].sum() > inst.B + tol:
        return False
    if inst.h[y].sum() > inst.H + tol:
        return False
    for k in range(len(inst.Q)):
        if np.isfinite(inst.Q[k]):
            sel = used & (MODEL_PROVIDER[np.maximum(a, 0)] == k)
            if inst.c[np.nonzero(sel)[0], a[sel]].sum() > inst.Q[k] + tol:
                return False
    if inst.fixed:
        return bool(used.all())
    if not used[inst.sinks].all():
        return False
    for i, kids in inst.and_groups.items():
        if any(used[j] != used[i] for j in kids):
            return False
    for i, kids in inst.or_groups.items():
        if sum(used[j] for j in kids) != int(used[i]):
            return False
    # a used task must be reachable from a deliverable through used groups
    reach = np.zeros(inst.n, dtype=bool)
    stack = [int(r) for r in inst.sinks]
    while stack:
        i = stack.pop()
        reach[i] = True
        for j in inst.and_groups.get(i, []) + inst.or_groups.get(i, []):
            if used[j]:
                stack.append(j)
    return bool(np.array_equal(reach, used))


In [ ]:
%%writecode Implementation/kp_solve.py
"""Gurobi solvers for the extended formulation: routing with spot checks
(ROUTE), the mean-value plan (EV), the extensive form with optional
systemic-failure cap and per-scenario reviews (the PI bound), and
branch-and-Benders-cut with closed-form cuts for fixed workflows.

Every solver takes one Gurobi environment, so a worker process checks out
its license once and reuses it for all of its jobs.
"""
from __future__ import annotations

import json
import math
import os
import time

import gurobipy as gp
import numpy as np
import scipy.sparse as sp
from gurobipy import GRB

import kp_cuts as kc
import kp_model as km


def make_env(threads: int | None = None) -> gp.Env:
    env = gp.Env(empty=True)
    env.setParam("OutputFlag", 0)
    if threads:
        env.setParam("Threads", int(threads))
    env.start()
    return env


def apply_params(m: gp.Model, opts: dict, time_limit: float):
    """opts: threads, mip_gap, soft_mem_gb, nodefile_start, nodefile_dir,
    seed, mip_focus, log_file."""
    m.Params.Threads = int(opts.get("threads", 4))
    m.Params.MIPGap = float(opts.get("mip_gap", 0.01))
    m.Params.TimeLimit = max(1.0, float(time_limit))
    if opts.get("soft_mem_gb"):
        m.Params.SoftMemLimit = float(opts["soft_mem_gb"])
    if opts.get("nodefile_start"):
        m.Params.NodefileStart = float(opts["nodefile_start"])
    if opts.get("nodefile_dir"):
        m.Params.NodefileDir = str(opts["nodefile_dir"])
    if opts.get("seed") is not None:
        m.Params.Seed = int(opts["seed"])
    if opts.get("mip_focus") is not None:
        m.Params.MIPFocus = int(opts["mip_focus"])
    if opts.get("log_file"):
        m.Params.LogFile = str(opts["log_file"])
        m.Params.OutputFlag = 1
        m.Params.LogToConsole = 0


GAP_LEVELS = (0.20, 0.10, 0.05, 0.02, 0.01)


class Tracker:
    """Callback state: the first time the gap reaches the target and each of
    GAP_LEVELS, a sparse trace of (seconds, incumbent, bound), and a progress
    file the monitor reads."""

    def __init__(self, target: float, label: str = "", progress_path=None, every: float = 10.0):
        self.target, self.label, self.path, self.every = target, label, progress_path, every
        self.t_target = None
        self.t_gap = {f"{g:g}": None for g in GAP_LEVELS}
        self.trace = []
        self.last = -1e9
        self.last_write = -1e9

    def note(self, t, best, bound, force=False):
        gap = abs(best - bound) / max(abs(best), 1e-10) if best < GRB.INFINITY else float("inf")
        if self.t_target is None and gap <= self.target + 1e-12:
            self.t_target = float(t)
        for g in GAP_LEVELS:
            if self.t_gap[f"{g:g}"] is None and gap <= g + 1e-12:
                self.t_gap[f"{g:g}"] = float(t)
        if force or t - self.last >= self.every:
            self.trace.append((round(float(t), 2), float(best) if best < GRB.INFINITY else None, float(bound)))
            self.last = t
        if self.path and t - self.last_write >= 30.0:
            self.last_write = t
            try:
                tmp = str(self.path) + ".tmp"
                with open(tmp, "w") as f:
                    json.dump(dict(plan=self.label, seconds=round(float(t), 1),
                                   incumbent=float(best) if best < GRB.INFINITY else None,
                                   bound=float(bound), gap=gap if math.isfinite(gap) else None,
                                   written=time.strftime("%Y-%m-%dT%H:%M:%S")), f)
                os.replace(tmp, self.path)
            except OSError:
                pass

    def __call__(self, model, where):
        if where == GRB.Callback.MIP:
            self.note(model.cbGet(GRB.Callback.RUNTIME), model.cbGet(GRB.Callback.MIP_OBJBST),
                      model.cbGet(GRB.Callback.MIP_OBJBND))
        elif where == GRB.Callback.MIPSOL:
            self.note(model.cbGet(GRB.Callback.RUNTIME), model.cbGet(GRB.Callback.MIPSOL_OBJBST),
                      model.cbGet(GRB.Callback.MIPSOL_OBJBND), force=True)


def pair_mode_for(q) -> str:
    return "z" if km.same_diff(np.asarray(q)) is not None else "w"


# ---------------------------------------------------------------- first stage
def first_stage(m: gp.Model, inst: km.Instance, mode: str, reviews: bool = True, use_fixed=None):
    """x (n, M), y (n), u (n or None), pair variables z (nA) or w (nA, M, M).

    use_fixed fixes the set of used tasks of a template (ROUTE's default
    decomposition); None lets a template choose its decomposition."""
    n, M = inst.n, inst.M
    x = m.addVars(n, M, vtype=GRB.BINARY, name="x")
    y = m.addVars(n, vtype=GRB.BINARY, name="y") if reviews else None
    u = None
    if inst.fixed:
        m.addConstrs((x.sum(i, "*") == 1 for i in range(n)), name="assign")
    elif use_fixed is not None:
        uf = np.asarray(use_fixed).astype(bool)
        m.addConstrs((x.sum(i, "*") == int(uf[i]) for i in range(n)), name="assign")
        if reviews:
            for i in range(n):
                if not uf[i]:
                    y[i].UB = 0.0
    else:
        u = m.addVars(n, vtype=GRB.BINARY, name="u")
        for r in inst.sinks:
            u[int(r)].LB = 1.0
        for i, kids in inst.and_groups.items():
            for j in kids:
                m.addConstr(u[j] == u[i], name=f"and[{i},{j}]")
        for i, kids in inst.or_groups.items():
            m.addConstr(gp.quicksum(u[j] for j in kids) == u[i], name=f"or[{i}]")
        m.addConstrs((x.sum(i, "*") == u[i] for i in range(n)), name="assign")
        if reviews:
            m.addConstrs((y[i] <= u[i] for i in range(n)), name="review_used")
    cost = gp.quicksum(inst.c[i, k] * x[i, k] for i in range(n) for k in range(M))
    m.addConstr(cost <= inst.B, name="budget")
    if reviews:
        m.addConstr(gp.quicksum(inst.h[i] * y[i] for i in range(n)) <= inst.H, name="review")
    for k in range(len(inst.Q)):
        if np.isfinite(inst.Q[k]):
            mods = [mm for mm in range(M) if km.MODEL_PROVIDER[mm] == k]
            m.addConstr(gp.quicksum(inst.c[i, mm] * x[i, mm] for i in range(n) for mm in mods) <= inst.Q[k],
                        name=f"quota[{k}]")
    if mode == "z":
        z = m.addVars(inst.nA, lb=0.0, ub=1.0, name="z")
        for a, (j, i) in enumerate(inst.arcs):
            for k in range(M):
                m.addConstr(z[a] >= x[j, k] + x[i, k] - 1, name=f"same_lo[{a},{k}]")
                m.addConstr(z[a] <= 1 - x[j, k] + x[i, k], name=f"same_hi[{a},{k}]")
        pair = z
    else:
        w = m.addVars(inst.nA, M, M, lb=0.0, ub=1.0, name="w")
        for a, (j, i) in enumerate(inst.arcs):
            for k in range(M):
                m.addConstr(w.sum(a, k, "*") <= x[j, k], name=f"w_row[{a},{k}]")
                m.addConstr(w.sum(a, "*", k) <= x[i, k], name=f"w_col[{a},{k}]")
            uj = 1 if u is None else u[j]
            ui = 1 if u is None else u[i]
            if use_fixed is not None:
                uj, ui = int(use_fixed[j]), int(use_fixed[i])
            m.addConstr(w.sum(a, "*", "*") >= uj + ui - 1, name=f"w_cover[{a}]")
        pair = w
    return x, y, u, pair


def set_start(inst, x, y, u, pair, mode, a, yv):
    a = np.asarray(a, dtype=int)
    for i in range(inst.n):
        for k in range(inst.M):
            x[i, k].Start = 1.0 if a[i] == k else 0.0
        if y is not None:
            y[i].Start = float(bool(yv[i]))
        if u is not None:
            u[i].Start = float(a[i] >= 0)
    pv = kc.pair_values(inst, a, mode)
    if mode == "z":
        for k in range(inst.nA):
            pair[k].Start = float(pv[k])
    else:
        for k in range(inst.nA):
            for p_ in range(inst.M):
                for q_ in range(inst.M):
                    pair[k, p_, q_].Start = float(pv[k, p_, q_])


def extract(inst, x, y, u=None):
    a = np.full(inst.n, -1, dtype=int)
    for i in range(inst.n):
        vals = [x[i, k].X for k in range(inst.M)]
        if max(vals) > 0.5:
            a[i] = int(np.argmax(vals))
    yv = np.array([y[i].X > 0.5 for i in range(inst.n)]) if y is not None else np.zeros(inst.n, bool)
    return a, yv


def _possible(inst: km.Instance, sc: km.Scenarios) -> np.ndarray:
    """(S, n): output i can be released wrong in scenario s under some plan."""
    poss = np.zeros((sc.S, inst.n), dtype=bool)
    own_any = sc.xi.any(axis=2)
    live = sc.U < sc.q.max()
    for i in range(inst.n):
        val = own_any[:, i].copy()
        for k in inst.in_arcs[i]:
            val |= poss[:, inst.arcs[k, 0]] & live[:, k]
        poss[:, i] = val
    return poss


def _result(m, method, t0, build, tracker, inst, x, y, u, extra=None):
    out = dict(method=method, status=int(m.Status), build_time=build, time=time.time() - t0,
               rows=int(m.NumConstrs), cols=int(m.NumVars), binaries=int(m.NumBinVars),
               nodes=float(m.NodeCount), time_to_target=tracker.t_target if tracker else None,
               time_to_gap=tracker.t_gap if tracker else {}, trace=tracker.trace if tracker else [],
               time_limit=float(m.Params.TimeLimit))
    out["bound"] = float(m.ObjBound) if (m.SolCount or m.Status == GRB.OPTIMAL) else float("nan")
    if m.SolCount:
        out["obj"], out["gap"] = float(m.ObjVal), float(m.MIPGap)
        if x is not None:
            out["a"], out["y"] = extract(inst, x, y, u)
    else:
        out["obj"], out["gap"] = float("nan"), float("nan")
    if extra:
        out.update(extra)
    return out


# ------------------------------------------------------------ extensive form
def solve_ef(inst: km.Instance, sc: km.Scenarios, lam: float, alpha: float, env: gp.Env, opts: dict,
             time_limit: float, start=None, cap_beta: float | None = None, per_scenario_reviews: bool = False,
             label: str = "EF", progress_path=None) -> dict:
    """Extensive form of the extended program on sample sc.

    cap_beta adds binary f_s with sum_i e_si <= E_sys - 1 + (n - E_sys + 1) f_s
    and sum_s f_s <= floor(beta |Omega|). per_scenario_reviews replaces y by
    y_s (S, n) under the same hours in every scenario: the perfect-information
    bound on review policies (its objective is a bound, not a plan)."""
    t0 = time.time()
    S, n, M = sc.S, inst.n, inst.M
    mode = pair_mode_for(sc.q)
    m = gp.Model(label, env=env)
    x, y, u, pair = first_stage(m, inst, mode, reviews=not per_scenario_reviews)
    if per_scenario_reviews:
        ys = m.addVars(S, n, vtype=GRB.BINARY, name="ys")
        for s in range(S):
            m.addConstr(gp.quicksum(inst.h[i] * ys[s, i] for i in range(n)) <= inst.H)
            if u is not None:
                for i in range(n):
                    m.addConstr(ys[s, i] <= u[i])
    m.update()
    xcol = np.array([[x[i, k].index for k in range(M)] for i in range(n)])
    if per_scenario_reviews:
        ycol_s = np.array([[ys[s, i].index for i in range(n)] for s in range(S)])
    else:
        ycol = np.array([y[i].index for i in range(n)])
    poss = _possible(inst, sc)
    ecol = -np.ones((S, n), dtype=np.int64)
    ne = int(poss.sum())
    base = m.NumVars
    ecol[poss] = base + np.arange(ne)
    obj_e = np.broadcast_to(inst.ell[None, :], (S, n))[poss]
    m.addMVar(ne, lb=0.0, ub=1.0, obj=(1 - lam) / S * obj_e, name="e")
    use_cvar = lam > 0
    if use_cvar:
        eta = m.addVar(lb=0.0, ub=float(inst.ell.sum()), obj=lam, name="eta")
        m.addMVar(S, lb=0.0, obj=lam / ((1 - alpha) * S), name="r")
    m.update()
    rows, cols, vals, rhs, sense = [], [], [], [], []
    r = 0

    def ycols(s_idx, i_idx):
        return ycol_s[s_idx, i_idx] if per_scenario_reviews else ycol[i_idx]

    # own-error rows: e_si + delta_si y_i - sum_m xi_sim x_im >= 0
    ss, ii, mm = np.nonzero(sc.xi)
    key = ss * n + ii
    uniq, inv = np.unique(key, return_inverse=True)
    us, ui = uniq // n, uniq % n
    nr = len(uniq)
    dl = sc.delta[us, ui]
    rows += [r + np.arange(nr), r + np.nonzero(dl)[0], r + inv]
    cols += [ecol[us, ui], ycols(us[dl], ui[dl]), xcol[ii, mm]]
    vals += [np.ones(nr), np.ones(int(dl.sum())), -np.ones(len(inv))]
    rhs.append(np.zeros(nr))
    r += nr
    always = sc.U < sc.q.min()
    live = sc.U < sc.q.max()
    sd = km.same_diff(sc.q)
    for k, (j, i) in enumerate(inst.arcs):
        valid = poss[:, j] & live[:, k]
        sa = np.nonzero(valid & always[:, k])[0]              # e_i - e_j + delta y_i >= 0
        na = len(sa)
        da = sc.delta[sa, i]
        rows += [r + np.arange(na), r + np.arange(na), r + np.nonzero(da)[0]]
        cols += [ecol[sa, i], ecol[sa, j], ycols(sa[da], np.full(int(da.sum()), i))]
        vals += [np.ones(na), -np.ones(na), np.ones(int(da.sum()))]
        rhs.append(np.zeros(na))
        r += na
        sb = np.nonzero(valid & ~always[:, k])[0]
        nb = len(sb)
        if nb == 0:
            continue
        db = sc.delta[sb, i]
        rows += [r + np.arange(nb), r + np.arange(nb), r + np.nonzero(db)[0]]
        cols += [ecol[sb, i], ecol[sb, j], ycols(sb[db], np.full(int(db.sum()), i))]
        vals += [np.ones(nb), -np.ones(nb), np.ones(int(db.sum()))]
        if mode == "z":
            qs, qd = sd
            ks = (sc.U[sb, k] < qs).astype(float)
            kd = (sc.U[sb, k] < qd).astype(float)
            # e_i - e_j + delta y_i - (ks - kd) z >= kd - 1
            rows.append(r + np.arange(nb))
            cols.append(np.full(nb, pair[k].index))
            vals.append(-(ks - kd))
            rhs.append(kd - 1.0)
        else:
            # e_i - e_j + delta y_i - sum_{passing pairs} w >= -1
            kap = sc.U[sb, k][:, None, None] < sc.q[None, :, :]
            s_i, p_i, q_i = np.nonzero(kap)
            rows.append(r + s_i)
            cols.append(np.array([pair[k, int(p_), int(q_)].index for p_, q_ in zip(p_i, q_i)], dtype=np.int64))
            vals.append(-np.ones(len(s_i)))
            rhs.append(-np.ones(nb))
        r += nb
    ge_rows = r
    if use_cvar:   # r_s + eta - sum_i ell_i e_si >= 0
        rcol = base + ne + 1 + np.arange(S)
        etacol = base + ne
        ps, pi = np.nonzero(poss)
        wsel = inst.ell[pi] != 0
        rows += [r + np.arange(S), r + np.arange(S), r + ps[wsel]]
        cols += [rcol, np.full(S, etacol), ecol[ps[wsel], pi[wsel]]]
        vals += [np.ones(S), np.ones(S), -inst.ell[pi[wsel]]]
        rhs.append(np.zeros(S))
        r += S
    A = sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))),
                      shape=(r, m.NumVars))
    m.addMConstr(A, None, GRB.GREATER_EQUAL, np.concatenate(rhs))
    f = None
    if cap_beta is not None:
        cnt = poss.sum(axis=1)
        risky = np.nonzero(cnt >= inst.e_sys)[0]
        big = inst.n - inst.e_sys + 1
        f = m.addVars(len(risky), vtype=GRB.BINARY, name="f")
        m.update()
        vlist = m.getVars()
        for t_, s in enumerate(risky):
            cols_s = ecol[s, poss[s]]
            m.addConstr(gp.LinExpr([1.0] * len(cols_s), [vlist[c] for c in cols_s]) - big * f[t_]
                        <= inst.e_sys - 1, name=f"sys[{s}]")
        m.addConstr(f.sum() <= math.floor(cap_beta * S + 1e-9), name="cap")
    build = time.time() - t0
    if start is not None:
        a0, y0 = start
        set_start(inst, x, y, u, pair, mode, a0, y0)
        e0 = km.propagate(inst, a0, y0, sc)
        evars = m.getVars()
        for (s, i) in zip(*np.nonzero(poss)):
            evars[ecol[s, i]].Start = float(e0[s, i])
        if per_scenario_reviews:
            for s in range(S):
                for i in range(n):
                    ys[s, i].Start = float(bool(y0[i]))
        if f is not None:
            cnt0 = e0.sum(axis=1)
            for t_, s in enumerate(risky):
                f[t_].Start = float(cnt0[s] >= inst.e_sys)
    apply_params(m, opts, time_limit - build)
    tracker = Tracker(float(opts.get("mip_gap", 0.01)), label, progress_path)
    m.optimize(tracker)
    extra = dict(pair_mode=mode, ge_rows=ge_rows)
    if per_scenario_reviews:
        return _result(m, label, t0, build, tracker, inst, None, None, None, extra)
    return _result(m, label, t0, build, tracker, inst, x, y, u, extra)


# ------------------------------------------------------- Benders, fixed workflow
def solve_benders(inst: km.Instance, sc: km.Scenarios, lam: float, alpha: float, env: gp.Env, opts: dict,
                  time_limit: float, start=None, warm_rounds: int = 60, warm_share: float = 0.25,
                  node_rounds: int = 20, node_every: int = 25, label: str = "BD", progress_path=None) -> dict:
    """Branch-and-Benders-cut with the closed-form cuts of kp_cuts.CutEngine:
    cutting planes on the master's linear relaxation, lazy cuts at integer
    solutions, user cuts at fractional nodes, and incumbents completed with
    their true scenario losses. Fixed workflows only; no cap."""
    if not inst.fixed:
        raise ValueError("Benders is implemented for fixed workflows")
    t0 = time.time()
    S, n, M = sc.S, inst.n, inst.M
    mode = pair_mode_for(sc.q)
    eng = kc.CutEngine(inst, sc, mode=mode)
    m = gp.Model(label, env=env)
    x, y, u, pair = first_stage(m, inst, mode)
    use_cvar = lam > 0
    th = m.addVars(S, lb=0.0, obj=(1 - lam) / S, name="theta")
    if use_cvar:
        eta = m.addVar(lb=0.0, ub=float(inst.ell.sum()), obj=lam, name="eta")
        tv = m.addVars(S, lb=0.0, obj=lam / ((1 - alpha) * S), name="r")
        m.addConstrs((tv[s] >= th[s] - eta for s in range(S)), name="cvar")
    m.update()
    xl = [[x[i, k] for k in range(M)] for i in range(n)]
    yl = [y[i] for i in range(n)]
    pkeys = list(range(inst.nA)) if mode == "z" else [(a, p_, q_) for a in range(inst.nA)
                                                      for p_ in range(M) for q_ in range(M)]
    pl = [pair[k] for k in pkeys]
    thl = [th[s] for s in range(S)]
    stats = dict(cuts=0, callbacks=0, warm_rounds=0, warm_cuts=0)

    def pvals(getter):
        v = np.array(getter(pl))
        return dict(zv=v) if mode == "z" else dict(wv=v.reshape(inst.nA, M, M))

    def expr(s, parent, Wt):
        xt, yt, pt, const = eng.cut(s, parent, Wt)
        ex = gp.LinExpr(const)
        if xt:
            ex.addTerms([c for (_, _, c) in xt], [xl[i][k] for (i, k, _) in xt])
        if yt:
            ex.addTerms([c for (_, c) in yt], [yl[i] for (i, _) in yt])
        if pt:
            if mode == "z":
                ex.addTerms([c for (_, c) in pt], [pair[k] for (k, _) in pt])
            else:
                ex.addTerms([c for (*_, c) in pt], [pair[k, a_, b_] for (k, a_, b_, _) in pt])
        return ex

    for i in range(n):
        yl[i].VType = GRB.CONTINUOUS
        for k in range(M):
            xl[i][k].VType = GRB.CONTINUOUS
    m.Params.Method = 1
    apply_params(m, opts, time_limit)
    lp_bound = float("nan")
    for rnd in range(warm_rounds):
        if time.time() - t0 > warm_share * time_limit:
            break
        m.Params.TimeLimit = max(1.0, warm_share * time_limit - (time.time() - t0))
        m.optimize()
        if m.Status != GRB.OPTIMAL:
            break
        lp_bound = float(m.ObjVal)
        xv = np.array([[xl[i][k].X for k in range(M)] for i in range(n)])
        yv = np.array([yl[i].X for i in range(n)])
        tvv = np.array([thl[s].X for s in range(S)])
        _, loss, parent, Wt = eng.evaluate(xv, yv, **pvals(lambda vs: [v.X for v in vs]))
        viol = np.nonzero(loss > tvv + 1e-6 * (1.0 + np.abs(loss)))[0]
        stats["warm_rounds"] = rnd + 1
        if len(viol) == 0:
            break
        for s in viol:
            m.addConstr(thl[s] >= expr(int(s), parent, Wt))
        stats["warm_cuts"] += len(viol)
    for i in range(n):
        yl[i].VType = GRB.BINARY
        for k in range(M):
            xl[i][k].VType = GRB.BINARY
    m.Params.Method = -1
    m.Params.LazyConstraints = 1
    m.Params.PreCrush = 1
    m.update()
    kq = min(S - 1, int(np.floor((1.0 - alpha) * S + 1e-9)))
    pend = dict(sol=None, best=float("inf"), root_rounds=0, repairs=0, node_cuts=0)
    xflat = [xl[i][k] for i in range(n) for k in range(M)]
    tl = [tv[s] for s in range(S)] if use_cvar else []

    def value(loss):
        return (1 - lam) * float(loss.mean()) + (lam * km.cvar(loss, alpha) if use_cvar else 0.0)

    def stash(a, yv, loss):
        val = value(loss)
        if val < pend["best"] - 1e-9:
            pend["best"] = val
            pv = kc.pair_values(inst, a, mode)
            vars_ = xflat + yl + pl + thl
            vals_ = list(np.eye(M)[a].ravel()) + [float(v) for v in yv] + list(np.ravel(pv)) + list(loss)
            if use_cvar:
                eta_v = float(np.sort(loss)[::-1][kq])
                vars_ = vars_ + [eta] + tl
                vals_ = vals_ + [eta_v] + list(np.maximum(loss - eta_v, 0.0))
            pend["sol"] = (vars_, vals_)

    if start is not None:
        a0, y0 = start
        set_start(inst, x, y, u, pair, mode, a0, y0)
        l0 = km.propagate(inst, a0, y0, sc) @ inst.ell
        for s in range(S):
            thl[s].Start = float(l0[s])
        pend["best"] = value(l0)
    tracker = Tracker(float(opts.get("mip_gap", 0.01)), label, progress_path)

    def cb(model, where):
        tracker(model, where)
        if where == GRB.Callback.MIPSOL:
            stats["callbacks"] += 1
            xv = np.round(np.array(model.cbGetSolution(xflat)).reshape(n, M))
            yv = np.round(np.array(model.cbGetSolution(yl)))
            tvv = np.array(model.cbGetSolution(thl))
            a = xv.argmax(axis=1)
            pv = kc.pair_values(inst, a, mode)
            _, loss, parent, Wt = eng.evaluate(xv, yv, **(dict(zv=pv) if mode == "z" else dict(wv=pv)))
            viol = np.nonzero(loss > tvv + 1e-6 * (1.0 + np.abs(loss)))[0]
            for s in viol:
                model.cbLazy(thl[s] >= expr(int(s), parent, Wt))
            stats["cuts"] += len(viol)
            if len(viol):
                stash(a, yv, loss)
        elif where == GRB.Callback.MIPNODE:
            if pend["sol"] is not None:
                vars_, vals_ = pend["sol"]
                model.cbSetSolution(vars_, vals_)
                model.cbUseSolution()
                pend["sol"] = None
                pend["repairs"] += 1
            if model.cbGet(GRB.Callback.MIPNODE_STATUS) != GRB.OPTIMAL:
                return
            nd = int(model.cbGet(GRB.Callback.MIPNODE_NODCNT))
            if nd == 0:
                if pend["root_rounds"] >= node_rounds:
                    return
                pend["root_rounds"] += 1
            elif nd % node_every != 0:
                return
            xv = np.array(model.cbGetNodeRel(xflat)).reshape(n, M)
            yv = np.array(model.cbGetNodeRel(yl))
            tvv = np.array(model.cbGetNodeRel(thl))
            _, loss, parent, Wt = eng.evaluate(xv, yv, **pvals(model.cbGetNodeRel))
            viol = np.nonzero(loss > tvv + 1e-3 * (1.0 + np.abs(loss)))[0]
            for s in viol:
                model.cbCut(thl[s] >= expr(int(s), parent, Wt))
            pend["node_cuts"] += len(viol)

    m.Params.TimeLimit = max(1.0, time_limit - (time.time() - t0))
    m.optimize(cb)
    extra = dict(pair_mode=mode, lp_bound=lp_bound, warm_rounds=stats["warm_rounds"], warm_cuts=stats["warm_cuts"],
                 cuts=stats["cuts"] + stats["warm_cuts"] + pend["node_cuts"], callbacks=stats["callbacks"],
                 node_cuts=pend["node_cuts"], repairs=pend["repairs"])
    return _result(m, label, t0, 0.0, tracker, inst, x, y, u, extra)


# ------------------------------------------------------------------ baselines
def solve_route(inst: km.Instance, env: gp.Env, threads: int = 2, use=None):
    """Routing with spot checks: models minimize the expected number of task
    errors under the budget and quotas; reviews then maximize the expected
    loss of own errors caught, p ell dbar, under the hours. A template keeps
    its default decomposition (first alternative of every OR group)."""
    n, M = inst.n, inst.M
    use = inst.default_use() if use is None else np.asarray(use).astype(bool)
    m = gp.Model("route", env=env)
    m.Params.Threads = threads
    x = m.addVars(n, M, vtype=GRB.BINARY)
    m.addConstrs(x.sum(i, "*") == int(use[i]) for i in range(n))
    cost = gp.quicksum(inst.c[i, k] * x[i, k] for i in range(n) for k in range(M))
    m.addConstr(cost <= inst.B)
    for k in range(len(inst.Q)):
        if np.isfinite(inst.Q[k]):
            mods = [mm for mm in range(M) if km.MODEL_PROVIDER[mm] == k]
            m.addConstr(gp.quicksum(inst.c[i, mm] * x[i, mm] for i in range(n) for mm in mods) <= inst.Q[k])
    m.setObjective(gp.quicksum(inst.p[i, k] * x[i, k] for i in range(n) for k in range(M)) + 1e-6 * cost)
    m.optimize()
    a = np.full(n, -1, dtype=int)
    for i in range(n):
        if use[i]:
            a[i] = max(range(M), key=lambda k: x[i, k].X)
    pa = np.where(use, inst.p[np.arange(n), np.maximum(a, 0)], 0.0)
    k2 = gp.Model("spot", env=env)
    k2.Params.Threads = threads
    y = k2.addVars(n, vtype=GRB.BINARY)
    for i in range(n):
        if not use[i]:
            y[i].UB = 0.0
    k2.addConstr(gp.quicksum(inst.h[i] * y[i] for i in range(n)) <= inst.H)
    k2.setObjective(gp.quicksum(pa[i] * inst.ell[i] * inst.dbar[i] * y[i] for i in range(n)), GRB.MAXIMIZE)
    k2.optimize()
    yv = np.array([y[i].X > 0.5 for i in range(n)])
    return a, yv


def solve_ev(inst: km.Instance, q, env: gp.Env, opts: dict, time_limit: float = 60.0, start=None):
    """Mean-value plan: every random event replaced by its mean. e_i is the
    expected error level of output i, a review lowers it by dbar_i, and a
    handoff passes it on with the mean probability of its model pair. With
    q_same >= q_diff and dbar = 1 these are the manuscript's rows."""
    n, M = inst.n, inst.M
    q = np.asarray(q, float)
    mode = pair_mode_for(q)
    m = gp.Model("EV", env=env)
    x, y, u, pair = first_stage(m, inst, mode)
    e = m.addVars(n, lb=0.0, ub=1.0, obj=list(inst.ell), name="e")
    for i in range(n):
        m.addConstr(e[i] >= gp.quicksum(inst.p[i, k] * x[i, k] for k in range(M)) - inst.dbar[i] * y[i])
    for a, (j, i) in enumerate(inst.arcs):
        if mode == "z":
            qs, qd = km.same_diff(q)
            if qd <= qs:
                m.addConstr(e[i] >= qd * e[j] - inst.dbar[i] * y[i])
            else:
                m.addConstr(e[i] >= qd * e[j] - pair[a] - inst.dbar[i] * y[i])
            m.addConstr(e[i] >= qs * e[j] - (1.0 - pair[a]) - inst.dbar[i] * y[i])
        else:
            for p_ in range(M):
                for q_ in range(M):
                    m.addConstr(e[i] >= q[p_, q_] * e[j] - (1.0 - pair[a, p_, q_]) - inst.dbar[i] * y[i])
    if start is not None:
        set_start(inst, x, y, u, pair, mode, *start)
    o = dict(opts)
    o.pop("log_file", None)
    apply_params(m, o, time_limit)
    m.Params.MIPGap = 1e-4
    m.optimize()
    a_, y_ = extract(inst, x, y, u)
    return a_, y_, dict(status=int(m.Status), gap=float(m.MIPGap) if m.SolCount else float("nan"),
                        time=float(m.Runtime))


## Check and plan

In [ ]:
def run(*args):
    subprocess.run([sys.executable, *args], cwd=ROOT, check=True)

run("-m", "unittest", "discover", "-s", "Verification", "-p", "test_code_only.py")
runner = "Implementation/kp_campaign.py"
run(runner, "plan", "--workers", "11", "--threads", "2")

## Run the experiments

The commands use the available local machine and Gurobi license. After the full run, execute the analysis command in the README.

In [ ]:
if PREFLIGHT:
    run(runner, "preflight")
if LAUNCH:
    run(runner, "launch")